# RAG-ассистент поддержки «Фильмоскоп»

Проектный notebook для доработки после ревью. Целевая структура соответствует заданию спринта 2: аудит и подготовка корпуса → чанкинг и эмбеддинги → поисковая инфраструктура и retrieval → диагноз и улучшение retrieval → генерация ответов → оценка качества и диагностика.

Текущие ячейки содержат ранее собранный baseline/improved RAG pipeline и сохранённые outputs. В следующих этапах доработки нужно пересобрать индексируемый корпус без заглушек, архива и дублей, перемерить финальную retrieval-схему, заменить бинарные faithfulness/answer relevance на шкалу 1–5 и выполнить полный eval на 66 запросах с golden-context comparison.


## Общие настройки, константы и импорты



In [2]:
import json
import re
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
DATA_PATH = ROOT / "filmoskop_kb.json"
EMBED_MODEL = "intfloat/multilingual-e5-base"
RECALL_KS = [1, 3, 5, 10, 20]
TOP_K = 20

### Проверка зависимостей



In [4]:
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "Не найдена зависимость: langchain-text-splitters. "
        "Установите её в активное окружение/Jupyter kernel: "
        "python -m pip install langchain-text-splitters"
    ) from e

from bs4 import BeautifulSoup
from ftfy import fix_text
import trafilatura
from sentence_transformers import SentenceTransformer
import faiss
import torch

print("Версия FAISS:", getattr(faiss, "__version__", "неизвестно"))

FAISS_GPU_AVAILABLE = (
    hasattr(faiss, "StandardGpuResources")
    and hasattr(faiss, "get_num_gpus")
    and faiss.get_num_gpus() > 0
)

if FAISS_GPU_AVAILABLE:
    print("Доступно GPU для FAISS:", faiss.get_num_gpus())
else:
    print("GPU FAISS недоступен; используется CPU FAISS.")

TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Устройство PyTorch для эмбеддингов:", TORCH_DEVICE)


Версия FAISS: 1.16.0
GPU FAISS недоступен; используется CPU FAISS.
Устройство PyTorch для эмбеддингов: cuda


## Этап 1. Аудит и подготовка корпуса

На этом этапе загружаем корпус и запросы, считаем базовую статистику и проверяем качество сырого текста: остатки HTML, HTML entities, mojibake, zero-width symbols, переносы через дефис, гомоглифы, заглушки, архивные/устаревшие разделы и дубли. Для очистки используем `ftfy`, `trafilatura`/`BeautifulSoup` для HTML и регулярную очистку Markdown/text.

Результат этапа — два набора документов:

- `clean_docs`: все 181 раздел базы знаний после нормализации текста, нужен для аудита и трассировки;
- `indexable_docs`: корпус для retrieval-index без архивных/устаревших материалов, заглушек, служебных/непубликуемых документов, слишком коротких неинформативных фрагментов и дублей.



In [6]:
with DATA_PATH.open(encoding="utf-8") as f:
    data = json.load(f)

docs = data["docs"]
queries = data["queries"]

print("Документы:", len(docs), "| Вопросы:", len(queries))
print("Форматы:", dict(Counter(d["format"] for d in docs)))
print("Категории:", dict(Counter(d["category"] for d in docs)))
print("Типы вопросов:", dict(Counter(q["kind"] for q in queries)))


Документы: 181 | Вопросы: 66
Форматы: {'text': 132, 'markdown': 28, 'html': 21}
Категории: {'Подписка и оплата': 38, 'Функции и контент': 34, 'Аккаунт и профили': 35, 'Устройства и качество': 42, 'Ошибки воспроизведения': 32}
Типы вопросов: {'exact': 15, 'semantic': 25, 'multi': 16, 'gap': 10}


In [7]:
HTML_NOISE_SELECTORS = [
    "script", "style", "noscript", "nav", "footer", "header",
    ".cookie-banner", ".breadcrumbs", ".main-menu",
]

QUALITY_PATTERNS = {
    "html_tags": re.compile(r"<[^>]+>"),
    "html_entities": re.compile(r"&(?:[a-zA-Z]+|#\d+|#x[0-9a-fA-F]+);"),
    "replacement_char": re.compile("\ufffd"),
    "latin_mojibake": re.compile(r"[\u00c3\u00c2][\x80-\xbf]|[\u00d0\u00d1][\x80-\xbf]"),
    "cyrillic_latin1_mojibake": re.compile(r"[\u00d0\u00d1][\u0080-\u00ff]"),
    "zero_width": re.compile(r"[\u200b\u200c\u200d\ufeff]"),
    "hyphen_linebreak": re.compile(r"(?<=[А-Яа-яЁёA-Za-z])-\s+(?=[А-Яа-яЁёA-Za-z])"),
}

QUALITY_ISSUE_LABELS = {
    "html_tags": "HTML-теги остались в тексте",
    "html_entities": "HTML-сущности вида &amp; или &#...;",
    "replacement_char": "символ замены � из-за повреждения источника",
    "latin_mojibake": "mojibake после неверной латинской/UTF-8 декодировки",
    "cyrillic_latin1_mojibake": "кириллический mojibake после неверной Latin-1 декодировки",
    "zero_width": "невидимые zero-width символы",
    "hyphen_linebreak": "перенос слова через дефис и разрыв строки",
}

EXCLUSION_REASON_LABELS = {
    "archive_or_outdated": "архивные или устаревшие материалы",
    "duplicate_or_near_duplicate": "дубликаты или почти дубликаты сохранённых документов",
    "irrecoverable_replacement_char": "невосстановимое повреждение текста символом �",
    "stub_service_or_non_support": "служебные, черновые или не относящиеся к поддержке материалы",
    "too_short_noninformative": "слишком короткий и неинформативный документ",
}

ZERO_WIDTH_RE = QUALITY_PATTERNS["zero_width"]
HYPHEN_LINEBREAK_RE = QUALITY_PATTERNS["hyphen_linebreak"]
HOMOGLYPH_TRANSLATION = str.maketrans({
    "Α": "A", "Β": "B", "Ε": "E", "Κ": "K", "Μ": "M", "Н": "H", "Ο": "O", "Ρ": "P",
    "С": "C", "Τ": "T", "Χ": "X", "а": "a", "е": "e", "о": "o", "р": "p", "с": "c",
    "х": "x", "у": "y",
})
DUPLICATE_KEEP_MAP = {
    95: 0,
    160: 23,
    121: 7,
    151: 83,
    77: 175,
    105: 158,
    86: 146,
}


def count_quality_issues(items, text_key: str):
    """Считает документы с типовыми проблемами качества текста.

    Args:
        items: Список словарей с `doc_id` и текстовым полем.
        text_key: Имя поля, в котором находится проверяемый текст.

    Returns:
        `pandas.DataFrame` с числом документов по каждому типу проблемы и примерами `doc_id`.
    """
    rows = []
    for issue, pattern in QUALITY_PATTERNS.items():
        hit_ids = [item["doc_id"] for item in items if pattern.search(item.get(text_key, "") or "")]
        rows.append({"issue": issue, "docs": len(hit_ids), "doc_ids_sample": hit_ids[:20]})
    return pd.DataFrame(rows)


def html_to_text(raw: str) -> str:
    """Извлекает основной текст из HTML-документа.

    Args:
        raw: Сырой HTML или текст с HTML-разметкой.

    Returns:
        Текст без служебной разметки, навигации, cookie-баннеров, скриптов и стилей.
    """
    extracted = trafilatura.extract(
        raw,
        include_comments=False,
        include_tables=False,
        favor_precision=True,
    )
    if extracted and len(extracted.strip()) >= 30:
        return extracted

    soup = BeautifulSoup(raw, "lxml")
    for selector in HTML_NOISE_SELECTORS:
        for node in soup.select(selector):
            node.decompose()
    main_node = soup.select_one("article.kb-article") or soup.select_one("main") or soup.body or soup
    return main_node.get_text(" ", strip=True)


def clean_markdown(text: str) -> str:
    """Удаляет базовую Markdown-разметку, сохраняя полезный текст.

    Args:
        text: Сырой markdown/plain text документ.

    Returns:
        Текст без ссылочной/структурной markdown-разметки.
    """
    text = re.sub(r"!\[[^\]]*\]\([^)]*\)", " ", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", text)
    text = re.sub(r"(?m)^\s{0,3}#{1,6}\s*", "", text)
    text = re.sub(r"(?m)^\s*[-*+]\s+", "", text)
    text = re.sub(r"[`*_>]+", " ", text)
    return text


def strip_synthetic_metadata_labels(text: str) -> str:
    """Удаляет служебные строки датасета из очищенного текста.

    Некоторые HTML/text документы после извлечения сохраняют строки вида
    `Заголовок: ...`, `Категория: ...`, `Текст: ...`. Это артефакт парсинга,
    поэтому он должен удаляться на этапе нормализации документов, а не в генераторе ответов.
    """
    return re.sub(
        r"(?im)^\s*(Заголовок|Категория|Текст)\s*:\s*.*(?:\n|$)",
        "",
        text,
    ).strip()


def normalize_safe_homoglyphs(text: str) -> str:
    """Исправляет безопасные гомоглифы только в латинских технических токенах.

    Замена намеренно узкая: она применяется к JSON/API/SQL-подобным словам, где кириллические
    символы выглядят как латиница. Русский текст не транслитерируется.
    """
    def replace_token(match):
        return match.group(0).translate(HOMOGLYPH_TRANSLATION)

    return re.sub(r"\b[A-Za-z0-9_\-]*[АВЕКМНОРСТХаеорсху][A-Za-z0-9_\-]*\b", replace_token, text)


def clean_text(raw: str, fmt: str = "") -> str:
    """Очищает документ и нормализует текст для retrieval.

    Args:
        raw: Исходный сырой документ из датасета.
        fmt: Формат документа из метаданных: `html`, `markdown` или `text`.

    Returns:
        Очищенный текст без HTML/Markdown-разметки, с исправлением mojibake, zero-width и переносов.
    """
    text = fix_text(raw or "")
    if fmt == "html" or re.search(r"<(?:html|body|div|p|article|main|h1|nav|br)\b", text, flags=re.IGNORECASE):
        text = html_to_text(text)
    text = strip_synthetic_metadata_labels(text)
    text = clean_markdown(text)
    text = fix_text(text)
    text = ZERO_WIDTH_RE.sub("", text)
    text = HYPHEN_LINEBREAK_RE.sub("", text)
    text = normalize_safe_homoglyphs(text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def exclusion_reason(doc):
    """Возвращает причину исключения документа из retrieval-index или `None`.

    Фильтр намеренно conservative: он удаляет только явные архивы/устаревшие статьи,
    заглушки, служебные/непубликуемые материалы, короткие неинформативные фрагменты,
    невосстановимо повреждённые тексты и заранее проверенные near-duplicates.
    """
    doc_id = doc["doc_id"]
    title = doc["title"]
    text = doc["text"]
    combined = f"{title}\n{text}"
    lower_combined = combined.lower()

    if doc_id in DUPLICATE_KEEP_MAP:
        return "duplicate_or_near_duplicate"
    if "�" in text:
        return "irrecoverable_replacement_char"
    title_lower = title.lower()
    text_prefix = text[:500].lower()
    if (
        re.search(r"\(архив|архив,|архив$", title_lower)
        or "статья устарела" in text_prefix
        or "актуальные условия" in text_prefix and "архив" in text_prefix
        or re.search(r"коды ошибок.*старая нумерация", title_lower)
    ):
        return "archive_or_outdated"
    if re.search(
        r"раздел (?:перенесён|в разработке)|статья обновляется|таблица совместимости|схема разделов|"
        r"служебное|проверка в[её]рстки|протокол встречи|регламент размещения|объявление|\bопрос\b|"
        r"player-api|экспорт обращений|черновик|не публиковать",
        lower_combined,
    ):
        return "stub_service_or_non_support"
    if len(text) < 180:
        return "too_short_noninformative"
    return None


raw_quality = count_quality_issues(docs, "raw")

clean_docs = []
for d in docs:
    clean_docs.append({
        "doc_id": d["doc_id"],
        "title": d["title"],
        "category": d["category"],
        "format": d.get("format", ""),
        "raw_len": len(d.get("raw", "") or ""),
        "text": clean_text(d["raw"], d.get("format", "")),
    })

for doc in clean_docs:
    doc["clean_len"] = len(doc["text"])

clean_quality = count_quality_issues(clean_docs, "text")
quality_comparison = raw_quality.merge(
    clean_quality,
    on="issue",
    suffixes=("_raw", "_clean"),
)

exclusion_rows = []
indexable_docs = []
for doc in clean_docs:
    reason = exclusion_reason(doc)
    if reason:
        exclusion_rows.append({
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "reason": reason,
            "clean_len": doc["clean_len"],
            "kept_doc_id": DUPLICATE_KEEP_MAP.get(doc["doc_id"]),
        })
    else:
        indexable_docs.append(doc)

exclusions_df = pd.DataFrame(exclusion_rows).sort_values(["reason", "doc_id"]).reset_index(drop=True)
corpus_profile = pd.DataFrame([
    {"metric": "raw_docs", "value": len(docs)},
    {"metric": "clean_docs", "value": len(clean_docs)},
    {"metric": "indexable_docs", "value": len(indexable_docs)},
    {"metric": "excluded_docs", "value": len(exclusions_df)},
    {"metric": "queries", "value": len(queries)},
    {"metric": "unique_relevant_docs", "value": len({doc_id for q in queries for doc_id in q["relevant_docs"]})},
])
length_profile = pd.Series([doc["clean_len"] for doc in clean_docs]).describe(percentiles=[0.1, 0.5, 0.9]).round(1)
relevant_doc_ids = {doc_id for q in queries for doc_id in q["relevant_docs"]}
excluded_relevant_docs = sorted(set(exclusions_df["doc_id"]) & relevant_doc_ids)

print("Базовый профиль корпуса")
display(corpus_profile)
print("Форматы:", dict(Counter(d["format"] for d in docs)))
print("Категории:", dict(Counter(d["category"] for d in docs)))
print("Типы вопросов:", dict(Counter(q["kind"] for q in queries)))
print("Длины очищенных документов")
display(length_profile.to_frame("clean_len"))
quality_comparison_display = quality_comparison.assign(
    issue_ru=quality_comparison["issue"].map(QUALITY_ISSUE_LABELS),
)[["issue", "issue_ru", "docs_raw", "doc_ids_sample_raw", "docs_clean", "doc_ids_sample_clean"]]
exclusion_summary = exclusions_df.groupby("reason").size().reset_index(name="docs")
exclusion_summary = exclusion_summary.assign(
    reason_ru=exclusion_summary["reason"].map(EXCLUSION_REASON_LABELS),
)[["reason", "reason_ru", "docs"]]

print("Сравнение качества: сырой текст против очищенного")
display(quality_comparison_display)
print("Исключения из retrieval-index по причинам")
display(exclusion_summary)
print("Исключённые документы, размеченные как relevant_docs:", excluded_relevant_docs)
print("Таблица исключений retrieval-index")
display(exclusions_df)

sample_issue_doc_ids = [20, 60, 75, 97, 120, 128, 151, 162, 167, 176]
cleaning_examples = []
for doc_id in sample_issue_doc_ids:
    raw_doc = next((d for d in docs if d["doc_id"] == doc_id), None)
    clean_doc = next((d for d in clean_docs if d["doc_id"] == doc_id), None)
    if raw_doc is None or clean_doc is None:
        continue
    cleaning_examples.append({
        "doc_id": doc_id,
        "format": raw_doc["format"],
        "title": raw_doc["title"],
        "raw_sample": raw_doc["raw"][:260].replace("\n", " "),
        "clean_sample": clean_doc["text"][:360],
    })

pd.DataFrame(cleaning_examples)






Базовый профиль корпуса


,metric,value
0,raw_docs,181
1,clean_docs,181
2,indexable_docs,134
3,excluded_docs,47
4,queries,66
5,unique_relevant_docs,43


Форматы: {'text': 132, 'markdown': 28, 'html': 21}
Категории: {'Подписка и оплата': 38, 'Функции и контент': 34, 'Аккаунт и профили': 35, 'Устройства и качество': 42, 'Ошибки воспроизведения': 32}
Типы вопросов: {'exact': 15, 'semantic': 25, 'multi': 16, 'gap': 10}
Длины очищенных документов


,clean_len
count,181.0
mean,708.9
std,422.4
min,14.0
10%,302.0
50%,570.0
90%,1211.0
max,1851.0


Сравнение качества: сырой текст против очищенного


,issue,issue_ru,docs_raw,doc_ids_sample_raw,docs_clean,doc_ids_sample_clean
0,html_tags,HTML-теги остались в тексте,21,"[8, 19, 23, 32, 47, 52, 57, 71, 74, 78, 87, 89...",0,[]
1,html_entities,HTML-сущности вида &amp; или &#...;,7,"[47, 99, 110, 147, 154, 155, 173]",0,[]
2,replacement_char,символ замены � из-за повреждения источника,1,[120],1,[120]
3,latin_mojibake,mojibake после неверной латинской/UTF-8 декоди...,16,"[22, 38, 43, 46, 54, 55, 60, 73, 93, 106, 111,...",0,[]
4,cyrillic_latin1_mojibake,кириллический mojibake после неверной Latin-1 ...,16,"[22, 38, 43, 46, 54, 55, 60, 73, 93, 106, 111,...",0,[]
5,zero_width,невидимые zero-width символы,3,"[128, 151, 176]",0,[]
6,hyphen_linebreak,перенос слова через дефис и разрыв строки,4,"[20, 97, 162, 167]",0,[]


Исключения из retrieval-index по причинам


,reason,reason_ru,docs
0,archive_or_outdated,архивные или устаревшие материалы,21
1,duplicate_or_near_duplicate,дубликаты или почти дубликаты сохранённых доку...,7
2,irrecoverable_replacement_char,невосстановимое повреждение текста символом �,1
3,stub_service_or_non_support,"служебные, черновые или не относящиеся к подде...",17
4,too_short_noninformative,слишком короткий и неинформативный документ,1


Исключённые документы, размеченные как relevant_docs: []
Таблица исключений retrieval-index


,doc_id,title,reason,clean_len,kept_doc_id
0,10,Вход на телевизоре по коду (архив),archive_or_outdated,315,NaN
1,17,Скорость интернета для просмотра (архив),archive_or_outdated,686,NaN
2,25,Тарифы «Фильмоскопа» (архив),archive_or_outdated,479,NaN
3,27,Совместный просмотр (архив),archive_or_outdated,599,NaN
4,28,Объёмный звук 5.1 (архив),archive_or_outdated,575,NaN
5,33,Привязанные устройства (архив),archive_or_outdated,298,NaN
6,45,Требования для просмотра в 4K (архив),archive_or_outdated,413,NaN
7,52,Раздел справки (архив),archive_or_outdated,97,NaN
8,65,"Коды ошибок плеера (архив, старая нумерация)",archive_or_outdated,417,NaN
9,67,Скачивание фильмов и сериалов (архив),archive_or_outdated,446,NaN


,doc_id,format,title,raw_sample,clean_sample
0,20,text,Просмотр за границей,Готовить что-то перед поездкой не нужно: аккау...,Готовить что-то перед поездкой не нужно: аккау...
1,60,text,Wi-Fi или кабель: как подключить телевизор,ï»¿Ð¡ÑÐ°Ð±Ð¸Ð»ÑÐ½Ð¾ÑÑÑ Ð¿ÑÐ¾ÑÐ¼Ð¾ÑÑÐ°...,"Стабильность просмотра сильно зависит от того,..."
2,75,text,Конфигурация player-api,"{""service"": ""player-api"", ""version"": ""2.14.3"",...","{""service"": ""player-api"", ""version"": ""2.14.3"",..."
3,97,text,История просмотров,История просмотров хранится отдельно для каждо...,История просмотров хранится отдельно для каждо...
4,120,text,Уведомления,«Фильмоскоп» присылает два типа уведомлений. С...,«Фильмоскоп» присылает два типа уведомлений. С...
5,128,text,Смена тарифа,﻿Перед переходом стоит сравнить условия: от та...,Перед переходом стоит сравнить условия: от тар...
6,151,text,Регистрация аккаунта,﻿Зарегистрироваться в «Фильмоскопе» можно по н...,Зарегистрироваться в «Фильмоскопе» можно по но...
7,162,markdown,Годовая подписка,# Годовая подписка Период оплаты выбирается п...,Годовая подписка Период оплаты выбирается при ...
8,167,text,Как измерить скорость интернета,Реальная скорость интернета на устройстве част...,Реальная скорость интернета на устройстве част...
9,176,text,Приложение вылетает на телевизоре,﻿Если приложение на телевизоре закрывается сам...,Если приложение на телевизоре закрывается само...


### Вывод по этапу 1

Корпус загружен полностью: 181 документ и 66 тестовых запросов. В аудите проверяются форматы, категории, длины документов, типы запросов и типовые дефекты текста: HTML/Markdown-разметка, HTML entities, mojibake-паттерны, zero-width symbols, переносы через дефис, гомоглифы и символы замены Unicode.

Очистка удаляет HTML/Markdown-артефакты, zero-width symbols, переносы через дефис и безопасные гомоглифы в технических токенах. Символы замены Unicode остаются невосстановимой потерей данных источника, поэтому такие документы не попадают в индекс.

Для дальнейшего retrieval используется `indexable_docs`: из него исключены явные архивные/устаревшие статьи, заглушки, служебные и непубликуемые материалы, слишком короткие неинформативные фрагменты и заранее проверенные дубли/near-duplicates с фиксацией `kept_doc_id`. Полный очищенный корпус `clean_docs` сохранён отдельно для трассировки и аудита.



## Этап 2. Чанкинг и эмбеддинги



In [10]:
CHUNK_STRATEGIES = {
    "recursive_250_50": {"kind": "recursive", "chunk_size": 250, "chunk_overlap": 50},
    "recursive_350_70": {"kind": "recursive", "chunk_size": 350, "chunk_overlap": 70},
    "recursive_500_100": {"kind": "recursive", "chunk_size": 500, "chunk_overlap": 100},
    "recursive_900_150": {"kind": "recursive", "chunk_size": 900, "chunk_overlap": 150},
    "sentence_300_1": {"kind": "sentence", "max_chars": 300, "overlap_sentences": 1},
    "sentence_420_1": {"kind": "sentence", "max_chars": 420, "overlap_sentences": 1},
    "sentence_620_1": {"kind": "sentence", "max_chars": 620, "overlap_sentences": 1},
    "semantic_500_1": {"kind": "semantic", "max_chars": 500, "overlap_sentences": 1},
    "semantic_700_1": {"kind": "semantic", "max_chars": 700, "overlap_sentences": 1},
    "full_doc": {"kind": "full_doc"},
}


def split_sentences(text: str):
    """Делит русский текст на предложения простой regex-эвристикой.

    Args:
        text: Очищенный текст документа.

    Returns:
        Список непустых предложений или весь текст одним элементом.
    """
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?…])\s+(?=[А-ЯЁA-Z0-9«])", text)
    return [sentence.strip() for sentence in sentences if sentence.strip()] or ([text] if text else [])


def make_chunk(page_content: str, doc, chunk_id: int):
    """Создаёт chunk-словарь с контентом и retrieval-метаданными.

    Args:
        page_content: Текст чанка.
        doc: Очищенный документ.
        chunk_id: Порядковый номер чанка внутри документа.

    Returns:
        Словарь чанка для индексации и retrieval.
    """
    return {
        "page_content": page_content.strip(),
        "doc_id": doc["doc_id"],
        "title": doc["title"],
        "category": doc["category"],
        "chunk_id": chunk_id,
    }


def make_recursive_chunks(clean_docs, chunk_size: int, chunk_overlap: int):
    """Строит `RecursiveCharacterTextSplitter` чанки с заданными параметрами.

    Args:
        clean_docs: Очищенные документы.
        chunk_size: Максимальная длина чанка в символах.
        chunk_overlap: Перекрытие чанков в символах.

    Returns:
        Список чанков с метаданными.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", "; ", ": ", " ", ""],
    )
    chunks = []
    for doc in clean_docs:
        source_text = f"{doc['title']}\n{doc['category']}\n{doc['text']}"
        for i, part in enumerate(splitter.split_text(source_text)):
            chunks.append(make_chunk(part, doc, i))
    return chunks


def make_sentence_window_chunks(clean_docs, max_chars: int, overlap_sentences: int = 1):
    """Строит короткие чанки по предложениям с sentence-overlap.

    Args:
        clean_docs: Очищенные документы.
        max_chars: Желаемая максимальная длина чанка.
        overlap_sentences: Число предложений из конца предыдущего чанка для overlap.

    Returns:
        Список sentence-based чанков.
    """
    chunks = []
    for doc in clean_docs:
        prefix = f"{doc['title']}. Категория: {doc['category']}. "
        sentences = split_sentences(doc["text"])
        start = 0
        chunk_id = 0
        while start < len(sentences):
            current = []
            size = len(prefix)
            end = start
            while end < len(sentences) and (not current or size + len(sentences[end]) + 1 <= max_chars):
                current.append(sentences[end])
                size += len(sentences[end]) + 1
                end += 1
            chunks.append(make_chunk(prefix + " ".join(current), doc, chunk_id))
            chunk_id += 1
            if end >= len(sentences):
                break
            start = max(start + 1, end - overlap_sentences)
    return chunks


def make_semantic_block_chunks(clean_docs, max_chars: int, overlap_sentences: int = 1):
    """Строит semantic-ish блоки по предложениям и маркерам смысловых границ.

    Args:
        clean_docs: Очищенные документы.
        max_chars: Желаемая максимальная длина смыслового блока.
        overlap_sentences: Число предложений overlap между соседними блоками.

    Returns:
        Список semantic-ish чанков: заголовок, категория и связный блок предложений.
    """
    boundary_re = re.compile(
        r"^(Важно|Шаг|Если |Для |Можно|Нельзя|Ошибка|Проблема|Решение|Ограничение|Подсказка)\b",
        flags=re.IGNORECASE,
    )
    chunks = []
    for doc in clean_docs:
        prefix = f"Заголовок: {doc['title']}\nКатегория: {doc['category']}\nТекст: "
        sentences = split_sentences(doc["text"])
        groups = []
        current = []
        size = len(prefix)
        for sentence in sentences:
            is_boundary = bool(boundary_re.search(sentence)) and current
            is_too_long = current and size + len(sentence) + 1 > max_chars
            if is_boundary or is_too_long:
                groups.append(current)
                current = current[-overlap_sentences:] if overlap_sentences else []
                size = len(prefix) + sum(len(item) + 1 for item in current)
            current.append(sentence)
            size += len(sentence) + 1
        if current:
            groups.append(current)
        for i, group in enumerate(groups):
            chunks.append(make_chunk(prefix + " ".join(group), doc, i))
    return chunks


def make_full_doc_chunks(clean_docs):
    """Создаёт один retrieval-чанк на документ.

    Args:
        clean_docs: Очищенные документы.

    Returns:
        Список document-level чанков.
    """
    return [
        make_chunk(
            f"Заголовок: {doc['title']}\nКатегория: {doc['category']}\nТекст: {doc['text']}",
            doc,
            0,
        )
        for doc in clean_docs
    ]


def make_chunks(clean_docs, strategy_config):
    """Диспетчеризует создание чанков по конфигурации стратегии.

    Args:
        clean_docs: Очищенные документы.
        strategy_config: Параметры стратегии из `CHUNK_STRATEGIES`.

    Returns:
        Список чанков выбранной стратегии.
    """
    kind = strategy_config["kind"]
    if kind == "recursive":
        return make_recursive_chunks(
            clean_docs,
            chunk_size=strategy_config["chunk_size"],
            chunk_overlap=strategy_config["chunk_overlap"],
        )
    if kind == "sentence":
        return make_sentence_window_chunks(
            clean_docs,
            max_chars=strategy_config["max_chars"],
            overlap_sentences=strategy_config["overlap_sentences"],
        )
    if kind == "semantic":
        return make_semantic_block_chunks(
            clean_docs,
            max_chars=strategy_config["max_chars"],
            overlap_sentences=strategy_config["overlap_sentences"],
        )
    if kind == "full_doc":
        return make_full_doc_chunks(clean_docs)
    raise ValueError(f"Неизвестный тип стратегии: {kind}")


chunk_sets = {}
chunk_stats = []
for name, params in CHUNK_STRATEGIES.items():
    chunk_sets[name] = make_chunks(indexable_docs, params)
    lengths = [len(c["page_content"]) for c in chunk_sets[name]]
    chunk_stats.append({
        "strategy": name,
        "chunks": len(lengths),
        "median_len": int(np.median(lengths)),
        "p90_len": int(np.quantile(lengths, 0.90)),
    })

pd.DataFrame(chunk_stats)



,strategy,chunks,median_len,p90_len
0,recursive_250_50,760,175,232
1,recursive_350_70,572,254,327
2,recursive_500_100,391,388,484
3,recursive_900_150,262,469,876
4,sentence_300_1,858,250,289
5,sentence_420_1,511,358,407
6,sentence_620_1,285,531,601
7,semantic_500_1,452,410,483
8,semantic_700_1,311,508,672
9,full_doc,134,848,1626


### Эмбеддинги и FAISS-индексы



In [12]:
embedder = SentenceTransformer(EMBED_MODEL, device=TORCH_DEVICE)

def encode_passages(texts):
    """Создаёт векторы документов для модели E5.

    Args:
        texts: Список текстов чанков.

    Returns:
        Массив эмбеддингов `float32`, нормализованных для поиска по
        скалярному произведению в FAISS.
    """
    vecs = embedder.encode(
        ["passage: " + t for t in texts],
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=True,
    )
    return np.asarray(vecs, dtype="float32")

def encode_queries(texts):
    """Создаёт нормализованные векторы запросов для модели E5.

    Args:
        texts: Список текстов запросов.

    Returns:
        Массив эмбеддингов `float32`, нормализованных для поиска по
        скалярному произведению в FAISS.
    """
    vecs = embedder.encode(
        ["query: " + t for t in texts],
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=False,
    )
    return np.asarray(vecs, dtype="float32")

def build_faiss_index(vectors):
    """Создаёт индекс FAISS `IndexFlatIP` для нормализованных эмбеддингов.

    Args:
        vectors: Матрица dense-эмбеддингов `float32` формы
            `(количество_чанков, размерность_эмбеддинга)`.

    Returns:
        Индекс FAISS с добавленными векторами. По умолчанию используется CPU
        FAISS; GPU FAISS используется только если он доступен в окружении.
    """
    cpu_index = faiss.IndexFlatIP(vectors.shape[1])
    if FAISS_GPU_AVAILABLE:
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
        index.add(vectors)
        return index

    cpu_index.add(vectors)
    return cpu_index

indexes = {}
embeddings = {}
for name, chunks in chunk_sets.items():
    embeddings[name] = encode_passages([c["page_content"] for c in chunks])
    indexes[name] = build_faiss_index(embeddings[name])
    print(name, embeddings[name].shape, "размер индекса", indexes[name].ntotal)


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

recursive_250_50 (760, 768) размер индекса 760


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

recursive_350_70 (572, 768) размер индекса 572


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

recursive_500_100 (391, 768) размер индекса 391


... (10 outputs left out)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

semantic_700_1 (311, 768) размер индекса 311


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

full_doc (134, 768) размер индекса 134


### Retrieval-метрики для сравнения стратегий чанкинга



In [14]:
eval_queries = [q for q in queries if q["kind"] != "gap"]
query_vecs = encode_queries([q["query"] for q in eval_queries])

def retrieved_doc_ids(strategy: str, q_vec, k: int):
    """Возвращает уникальные `doc_id` из top-k найденных чанков.

    Args:
        strategy: Название стратегии чанкинга из `chunk_sets` и `indexes`.
        q_vec: Вектор запроса.
        k: Количество чанков, запрашиваемое у FAISS.

    Returns:
        Список уникальных идентификаторов документов в порядке релевантности.
    """
    _, idx = indexes[strategy].search(q_vec.reshape(1, -1), k)
    doc_ids = []
    seen = set()
    for i in idx[0]:
        doc_id = chunk_sets[strategy][int(i)]["doc_id"]
        if doc_id not in seen:
            doc_ids.append(doc_id)
            seen.add(doc_id)
    return doc_ids

def precision_at_k(found, relevant):
    """Считает precision для найденных `doc_id`.

    Args:
        found: Список найденных идентификаторов документов.
        relevant: Список эталонных идентификаторов документов.

    Returns:
        Доля найденных документов, которые входят в список релевантных.
    """
    if not found:
        return 0.0
    rel = set(relevant)
    return sum(doc_id in rel for doc_id in found) / len(found)

def recall_at_k(found, relevant):
    """Считает recall для найденных `doc_id`.

    Args:
        found: Список найденных идентификаторов документов.
        relevant: Список эталонных идентификаторов документов.

    Returns:
        Доля релевантных документов, найденных поиском.
    """
    rel = set(relevant)
    if not rel:
        return 0.0
    return len(set(found) & rel) / len(rel)

def mrr(found, relevant):
    """Считает reciprocal rank первого релевантного документа.

    Args:
        found: Список найденных идентификаторов документов в порядке релевантности.
        relevant: Список эталонных идентификаторов документов.

    Returns:
        Значение reciprocal rank или `0.0`, если релевантных документов нет в выдаче.
    """
    rel = set(relevant)
    for rank, doc_id in enumerate(found, start=1):
        if doc_id in rel:
            return 1 / rank
    return 0.0

def evaluate_strategy(strategy: str):
    """Считает retrieval-метрики для одной стратегии чанкинга.

    Args:
        strategy: Название стратегии чанкинга из `chunk_sets` и `indexes`.

    Returns:
        `pandas.DataFrame` по вопросам со списками найденных и релевантных документов.
    """
    rows = []
    for q, q_vec in zip(eval_queries, query_vecs):
        found20 = retrieved_doc_ids(strategy, q_vec, TOP_K)
        row = {
            "strategy": strategy,
            "kind": q["kind"],
            "query": q["query"],
            "relevant_docs": q["relevant_docs"],
            "found_doc_ids": found20,
            "precision@5": precision_at_k(found20[:5], q["relevant_docs"]),
            "recall@5": recall_at_k(found20[:5], q["relevant_docs"]),
            "MRR": mrr(found20, q["relevant_docs"]),
        }
        for k in RECALL_KS:
            row[f"recall@{k}"] = recall_at_k(found20[:k], q["relevant_docs"])
        rows.append(row)
    return pd.DataFrame(rows)

metrics_by_query = pd.concat([evaluate_strategy(name) for name in chunk_sets], ignore_index=True)
summary = metrics_by_query.groupby("strategy")[["precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"]].mean().reset_index()
summary


,strategy,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
0,full_doc,0.185714,0.705357,0.564718,0.312500,0.607143,0.919643,0.982143
1,recursive_250_50,0.178571,0.669643,0.665676,0.473214,0.598214,0.821429,0.875000
2,recursive_350_70,0.182143,0.687500,0.586345,0.366071,0.535714,0.910714,0.910714
3,recursive_500_100,0.175000,0.660714,0.582647,0.392857,0.598214,0.767857,0.821429
4,recursive_900_150,0.157143,0.589286,0.510269,0.294643,0.482143,0.741071,0.857143
5,semantic_500_1,0.203571,0.785714,0.699463,0.464286,0.705357,0.839286,0.901786
6,semantic_700_1,0.196429,0.750000,0.677169,0.464286,0.651786,0.883929,0.928571
7,sentence_300_1,0.217857,0.785714,0.653323,0.437500,0.687500,0.848214,0.848214
8,sentence_420_1,0.196429,0.750000,0.694118,0.491071,0.660714,0.875000,0.875000
9,sentence_620_1,0.189286,0.714286,0.654500,0.446429,0.616071,0.919643,0.937500


In [15]:
by_kind = metrics_by_query.groupby(["strategy", "kind"])[["precision@5", "recall@5", "MRR"]].mean().reset_index()
by_kind

,strategy,kind,precision@5,recall@5,MRR
0,full_doc,exact,0.213333,1.000000,0.755556
1,full_doc,multi,0.250000,0.625000,0.545387
2,full_doc,semantic,0.128000,0.580000,0.462586
3,recursive_250_50,exact,0.213333,1.000000,1.000000
4,recursive_250_50,multi,0.262500,0.656250,0.765873
5,recursive_250_50,semantic,0.104000,0.480000,0.400956
6,recursive_350_70,exact,0.200000,0.966667,0.900000
7,recursive_350_70,multi,0.262500,0.656250,0.656845
8,recursive_350_70,semantic,0.120000,0.540000,0.353032
9,recursive_500_100,exact,0.200000,0.966667,0.950000


### Анализ примеров чанков и выбор стратегии



In [17]:
for name, chunks in chunk_sets.items():
    print("\n===", name, "===")
    for c in chunks[:3]:
        print(f"документ={c['doc_id']} чанк={c['chunk_id']} заголовок={c['title']}")
        print(c["page_content"][:450].replace("\n", " "))
        print("---")


=== recursive_250_50 ===
документ=0 чанк=0 заголовок=Промокоды
Промокоды Подписка и оплата
---
документ=0 чанк=1 заголовок=Промокоды
Промокод даёт скидку на подписку или бесплатные дни. Ввести его можно в разделе «Подписка» — поле «У меня есть промокод» — или при оформлении подписки
---
документ=0 чанк=2 заголовок=Промокоды
. Правила простые: один промокод на один аккаунт, скидки по промокодам не суммируются между собой и c другими акциями, y каждого кода есть срок действия — он указан в условиях акции
---

=== recursive_350_70 ===
документ=0 чанк=0 заголовок=Промокоды
Промокоды Подписка и оплата
---
документ=0 чанк=1 заголовок=Промокоды
Промокод даёт скидку на подписку или бесплатные дни. Ввести его можно в разделе «Подписка» — поле «У меня есть промокод» — или при оформлении подписки. Правила простые: один промокод на один аккаунт, скидки по промокодам не суммируются между собой и c другими акциями, y каждого кода есть срок действия — он указан в условиях акции
---
документ=0 чанк=2

In [18]:
best_strategy = summary.sort_values(["recall@5", "MRR", "precision@5"], ascending=False).iloc[0]["strategy"]
print("Выбранная стратегия чанкинга:", best_strategy)
print(summary.to_string(index=False))


Выбранная стратегия чанкинга: semantic_500_1
         strategy  precision@5  recall@5      MRR  recall@1  recall@3  recall@10  recall@20
         full_doc     0.185714  0.705357 0.564718  0.312500  0.607143   0.919643   0.982143
 recursive_250_50     0.178571  0.669643 0.665676  0.473214  0.598214   0.821429   0.875000
 recursive_350_70     0.182143  0.687500 0.586345  0.366071  0.535714   0.910714   0.910714
recursive_500_100     0.175000  0.660714 0.582647  0.392857  0.598214   0.767857   0.821429
recursive_900_150     0.157143  0.589286 0.510269  0.294643  0.482143   0.741071   0.857143
   semantic_500_1     0.203571  0.785714 0.699463  0.464286  0.705357   0.839286   0.901786
   semantic_700_1     0.196429  0.750000 0.677169  0.464286  0.651786   0.883929   0.928571
   sentence_300_1     0.217857  0.785714 0.653323  0.437500  0.687500   0.848214   0.848214
   sentence_420_1     0.196429  0.750000 0.694118  0.491071  0.660714   0.875000   0.875000
   sentence_620_1     0.189286  0.7

### Вывод по этапу 2

Этап 2 пересчитан на `indexable_docs`: в индексируемом корпусе 134 документа после исключения заглушек, архива, дублей и невосстановимо повреждённого текста. Сравнены 10 стратегий чанкинга на одном наборе из 56 non-gap запросов. Для эмбеддингов используется `intfloat/multilingual-e5-base` с префиксами E5 `passage:` для чанков и `query:` для запросов; векторы нормализованы.

Лучшая стратегия по `recall@5`, затем `MRR` и `precision@5` — `semantic_500_1`: 452 чанка, `precision@5=0.203571`, `recall@5=0.785714`, `MRR=0.699463`, `recall@20=0.901786`. Очень близкая по `recall@5` стратегия `sentence_300_1` даёт больше чанков (858) и ниже `MRR`, поэтому для baseline выбрана `semantic_500_1` как более компактная и стабильная точка сравнения.

Ручной просмотр примеров показывает, что semantic/sentence чанки сохраняют заголовок и категорию, а факты обычно не отрываются от смысла соседних предложений. Дальнейшее качество ограничено не только чанкингом: для части semantic и multi запросов релевантные документы появляются ниже top-5 или требуют объединения evidence из нескольких документов.


## Этап 3. Поисковая инфраструктура и retrieval

На выбранной стратегии чанкинга собираем итоговый dense baseline: выдачу top-K чанков, распределение score для подбора порога, doc-level overfetch с агрегацией чанков в уникальные `doc_id`, пример фильтрации по категории и диагностику ошибок по типам запросов.


In [21]:
BASELINE_CHUNK_TOP_K = 20
OVERFETCH_CHUNK_KS = [50, 100]
DOC_TOP_K = 20
FILTER_CATEGORY = "Подписка и оплата"

selected_strategy = best_strategy
selected_chunks = chunk_sets[selected_strategy]
selected_index = indexes[selected_strategy]
doc_by_id = {doc["doc_id"]: doc for doc in indexable_docs}

eval_query_rows = list(zip(eval_queries, query_vecs))

print("Стратегия для базового поиска:", selected_strategy)
print("Чанков в выбранной стратегии:", len(selected_chunks))
print("Оценочных запросов без gap:", len(eval_query_rows))

Стратегия для базового поиска: semantic_500_1
Чанков в выбранной стратегии: 452
Оценочных запросов без gap: 56


In [22]:
def dense_chunk_search(strategy: str, q_vec, chunk_k: int, category: str | None = None, score_threshold: float | None = None):
    """Ищет чанки через FAISS и возвращает результаты с метаданными.
    
        Args:
            strategy: Название стратегии чанкинга из `chunk_sets` и `indexes`.
            q_vec: Вектор запроса.
            chunk_k: Количество chunk-кандидатов, запрашиваемых у FAISS.
            category: Необязательная категория для постфильтрации результатов.
            score_threshold: Необязательный минимальный dense-score.
    
        Returns:
            Список словарей с rank, score, doc_id, title, category, chunk_id и page_content.
    """
    chunks = chunk_sets[strategy]
    k = min(chunk_k, len(chunks))
    scores, idx = indexes[strategy].search(q_vec.reshape(1, -1), k)
    hits = []
    rank = 0
    for score, chunk_idx in zip(scores[0], idx[0]):
        if chunk_idx < 0:
            continue
        chunk = chunks[int(chunk_idx)]
        score = float(score)
        if category is not None and chunk["category"] != category:
            continue
        if score_threshold is not None and score < score_threshold:
            continue
        rank += 1
        hits.append({
            "rank": rank,
            "chunk_index": int(chunk_idx),
            "score": score,
            "doc_id": chunk["doc_id"],
            "title": chunk["title"],
            "category": chunk["category"],
            "chunk_id": chunk["chunk_id"],
            "page_content": chunk["page_content"],
        })
    return hits


def aggregate_chunks_to_docs(chunk_hits, doc_k: int | None = None):
    """Агрегирует результаты уровня чанков в уникальные документы по лучшему score чанка.
    
        Args:
            chunk_hits: Результаты, вернутые `dense_chunk_search`.
            doc_k: Необязательное максимальное число документов в выдаче.
    
        Returns:
            Ранжированный список уникальных document hits.
    """
    docs_map = {}
    for hit in chunk_hits:
        doc_id = hit["doc_id"]
        if doc_id not in docs_map:
            docs_map[doc_id] = {
                "doc_id": doc_id,
                "title": hit["title"],
                "category": hit["category"],
                "best_score": hit["score"],
                "best_chunk_rank": hit["rank"],
                "best_chunk_id": hit["chunk_id"],
                "chunk_count": 1,
                "page_content": hit["page_content"],
            }
        else:
            docs_map[doc_id]["chunk_count"] += 1
            if hit["score"] > docs_map[doc_id]["best_score"]:
                docs_map[doc_id].update({
                    "best_score": hit["score"],
                    "best_chunk_rank": hit["rank"],
                    "best_chunk_id": hit["chunk_id"],
                    "page_content": hit["page_content"],
                })
    docs_ranked = sorted(
        docs_map.values(),
        key=lambda row: (-row["best_score"], row["best_chunk_rank"]),
    )
    if doc_k is not None:
        docs_ranked = docs_ranked[:doc_k]
    for rank, row in enumerate(docs_ranked, start=1):
        row["doc_rank"] = rank
    return docs_ranked


def dense_doc_ids_with_overfetch(strategy: str, q_vec, chunk_k: int, doc_k: int):
    """Возвращает top document id после overfetch chunk-кандидатов.
    
        Args:
            strategy: Название стратегии чанкинга из `chunk_sets` и `indexes`.
            q_vec: Вектор запроса.
            chunk_k: Количество chunk-кандидатов, запрашиваемых у FAISS.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            Ранжированный список уникальных идентификаторов документов.
    """
    chunk_hits = dense_chunk_search(strategy, q_vec, chunk_k)
    doc_hits = aggregate_chunks_to_docs(chunk_hits, doc_k=doc_k)
    return [row["doc_id"] for row in doc_hits]


def evaluate_dense_variant(strategy: str, chunk_k: int, doc_k: int, variant: str):
    """Считает retrieval-метрики для dense-варианта с выбранным overfetch.
    
        Args:
            variant: Человекочитаемое название варианта.
            chunk_k: Количество chunk-кандидатов, запрашиваемых у FAISS.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            `pandas.DataFrame` с метриками варианта по каждому запросу.
    """
    rows = []
    for q, q_vec in eval_query_rows:
        found_docs = dense_doc_ids_with_overfetch(strategy, q_vec, chunk_k=chunk_k, doc_k=doc_k)
        row = {
            "variant": variant,
            "strategy": strategy,
            "chunk_k": chunk_k,
            "doc_k": doc_k,
            "kind": q["kind"],
            "query": q["query"],
            "relevant_docs": q["relevant_docs"],
            "found_doc_ids": found_docs,
            "precision@5": precision_at_k(found_docs[:5], q["relevant_docs"]),
            "recall@5": recall_at_k(found_docs[:5], q["relevant_docs"]),
            "MRR": mrr(found_docs, q["relevant_docs"]),
        }
        for k in RECALL_KS:
            row[f"recall@{k}"] = recall_at_k(found_docs[:k], q["relevant_docs"])
        rows.append(row)
    return pd.DataFrame(rows)

In [23]:
score_rows = []
for query_idx, (q, q_vec) in enumerate(eval_query_rows):
    for hit in dense_chunk_search(selected_strategy, q_vec, chunk_k=BASELINE_CHUNK_TOP_K):
        score_rows.append({
            "query_idx": query_idx,
            "kind": q["kind"],
            "chunk_rank": hit["rank"],
            "score": hit["score"],
            "doc_id": hit["doc_id"],
            "is_relevant_doc": hit["doc_id"] in set(q["relevant_docs"]),
        })

score_distribution = pd.DataFrame(score_rows)
score_quantiles = score_distribution["score"].quantile([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).reset_index()
score_quantiles.columns = ["quantile", "score"]
DENSE_SCORE_THRESHOLD = float(score_distribution["score"].quantile(0.10))

print("Диагностический порог dense-score p10:", round(DENSE_SCORE_THRESHOLD, 4))
score_quantiles

Диагностический порог dense-score p10: 0.8038


,quantile,score
0,0.01,0.776652
1,0.05,0.797239
2,0.10,0.803833
3,0.25,0.816347
4,0.50,0.829610
5,0.75,0.843795
6,0.90,0.856823
7,0.95,0.863265
8,0.99,0.875640


In [24]:
sample_query, sample_vec = eval_query_rows[0]
sample_chunk_hits = dense_chunk_search(selected_strategy, sample_vec, chunk_k=10)
sample_doc_hits = aggregate_chunks_to_docs(sample_chunk_hits, doc_k=5)

print("Пример запроса:", sample_query["query"])
print("Эталонные doc_id:", sample_query["relevant_docs"])
pd.DataFrame(sample_chunk_hits)[["rank", "score", "doc_id", "title", "category", "chunk_id"]].head(10)

Пример запроса: что означает ошибка P-102
Эталонные doc_id: [85]


,rank,score,doc_id,title,category,chunk_id
0,1,0.872476,85,Ошибка P-102: превышен лимит просмотров,Ошибки воспроизведения,0
1,2,0.869149,85,Ошибка P-102: превышен лимит просмотров,Ошибки воспроизведения,1
2,3,0.867126,85,Ошибка P-102: превышен лимит просмотров,Ошибки воспроизведения,2
3,4,0.866126,178,Ошибка P-109: недостаточно места,Ошибки воспроизведения,0
4,5,0.865430,129,Ошибка P-105: устаревшее приложение,Ошибки воспроизведения,0
5,6,0.864898,137,Ошибка P-106: сбой загрузки субтитров,Ошибки воспроизведения,0
6,7,0.864189,137,Ошибка P-106: сбой загрузки субтитров,Ошибки воспроизведения,1
7,8,0.864061,85,Ошибка P-102: превышен лимит просмотров,Ошибки воспроизведения,3
8,9,0.862048,47,Ошибка P-103: контент недоступен в регионе,Ошибки воспроизведения,0
9,10,0.861484,47,Ошибка P-103: контент недоступен в регионе,Ошибки воспроизведения,1


In [15]:
print("Топ-5 документов после агрегации попаданий чанков:")
pd.DataFrame(sample_doc_hits)[["doc_rank", "best_score", "doc_id", "title", "category", "best_chunk_id", "chunk_count"]]

Топ-5 документов после агрегации попаданий чанков:


,doc_rank,best_score,doc_id,title,category,best_chunk_id,chunk_count
0,1,0.872476,85,Ошибка P-102: превышен лимит просмотров,Ошибки воспроизведения,0,4
1,2,0.866126,178,Ошибка P-109: недостаточно места,Ошибки воспроизведения,0,1
2,3,0.865430,129,Ошибка P-105: устаревшее приложение,Ошибки воспроизведения,0,1
3,4,0.864898,137,Ошибка P-106: сбой загрузки субтитров,Ошибки воспроизведения,0,2
4,5,0.862048,47,Ошибка P-103: контент недоступен в регионе,Ошибки воспроизведения,0,2


In [26]:
baseline_variants = []
baseline_variants.append(evaluate_dense_variant(
    selected_strategy,
    chunk_k=BASELINE_CHUNK_TOP_K,
    doc_k=DOC_TOP_K,
    variant="dense_chunk20_doc20",
))
for chunk_k in OVERFETCH_CHUNK_KS:
    baseline_variants.append(evaluate_dense_variant(
        selected_strategy,
        chunk_k=chunk_k,
        doc_k=DOC_TOP_K,
        variant=f"dense_chunk{chunk_k}_doc20",
    ))

baseline_diagnostics = pd.concat(baseline_variants, ignore_index=True)
baseline_summary = baseline_diagnostics.groupby("variant")[[
    "precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"
]].mean().reset_index().sort_values(["recall@20", "recall@5", "MRR"], ascending=False)
baseline_summary

,variant,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
0,dense_chunk100_doc20,0.203571,0.785714,0.700836,0.464286,0.705357,0.848214,0.982143
2,dense_chunk50_doc20,0.203571,0.785714,0.700836,0.464286,0.705357,0.848214,0.982143
1,dense_chunk20_doc20,0.203571,0.785714,0.699463,0.464286,0.705357,0.839286,0.901786


In [27]:
baseline_by_kind = baseline_diagnostics.groupby(["variant", "kind"])[[
    "precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"
]].mean().reset_index()
baseline_by_kind.sort_values(["variant", "kind"])

,variant,kind,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
0,dense_chunk100_doc20,exact,0.213333,1.00000,0.933333,0.833333,0.966667,1.00000,1.0000
1,dense_chunk100_doc20,multi,0.262500,0.65625,0.792140,0.343750,0.593750,0.81250,1.0000
2,dense_chunk100_doc20,semantic,0.160000,0.74000,0.502904,0.320000,0.620000,0.78000,0.9600
3,dense_chunk20_doc20,exact,0.213333,1.00000,0.933333,0.833333,0.966667,1.00000,1.0000
4,dense_chunk20_doc20,multi,0.262500,0.65625,0.792140,0.343750,0.593750,0.78125,0.8125
5,dense_chunk20_doc20,semantic,0.160000,0.74000,0.499827,0.320000,0.620000,0.78000,0.9000
6,dense_chunk50_doc20,exact,0.213333,1.00000,0.933333,0.833333,0.966667,1.00000,1.0000
7,dense_chunk50_doc20,multi,0.262500,0.65625,0.792140,0.343750,0.593750,0.81250,1.0000
8,dense_chunk50_doc20,semantic,0.160000,0.74000,0.502904,0.320000,0.620000,0.78000,0.9600


In [28]:
category_query_idx = None
for i, (q, q_vec) in enumerate(eval_query_rows):
    relevant_categories = {doc_by_id[doc_id]["category"] for doc_id in q["relevant_docs"] if doc_id in doc_by_id}
    if FILTER_CATEGORY in relevant_categories:
        category_query_idx = i
        break

if category_query_idx is None:
    category_query_idx = 0

category_query, category_vec = eval_query_rows[category_query_idx]
category_hits = dense_chunk_search(
    selected_strategy,
    category_vec,
    chunk_k=100,
    category=FILTER_CATEGORY,
    score_threshold=None,
)
category_doc_hits = aggregate_chunks_to_docs(category_hits, doc_k=5)

print("Категория фильтра:", FILTER_CATEGORY)
print("Запрос:", category_query["query"])
print("Эталонные doc_id:", category_query["relevant_docs"])
pd.DataFrame(category_doc_hits)[["doc_rank", "best_score", "doc_id", "title", "category", "best_chunk_id", "chunk_count"]]

Категория фильтра: Подписка и оплата
Запрос: сколько стоит тариф Максимум и что в него входит
Эталонные doc_id: [113]


,doc_rank,best_score,doc_id,title,category,best_chunk_id,chunk_count
0,1,0.863328,24,Какой тариф подойдёт именно вам,Подписка и оплата,0,4
1,2,0.856784,113,Тарифы «Фильмоскопа»,Подписка и оплата,3,7
2,3,0.833375,32,Почему зрители выбирают «Максимум»,Подписка и оплата,0,1
3,4,0.817628,29,Как меняются цены,Подписка и оплата,3,5
4,5,0.815394,128,Смена тарифа,Подписка и оплата,2,7


In [29]:
diagnostic_variant = "dense_chunk100_doc20" if "dense_chunk100_doc20" in set(baseline_diagnostics["variant"]) else baseline_summary.iloc[0]["variant"]
problem_rows = []

for q, q_vec in eval_query_rows:
    chunk_hits_100 = dense_chunk_search(selected_strategy, q_vec, chunk_k=100)
    doc_hits_100 = aggregate_chunks_to_docs(chunk_hits_100, doc_k=100)
    doc_positions = {row["doc_id"]: row["doc_rank"] for row in doc_hits_100}
    found20 = [row["doc_id"] for row in doc_hits_100[:DOC_TOP_K]]
    relevant = q["relevant_docs"]
    relevant_positions = {doc_id: doc_positions.get(doc_id) for doc_id in relevant}
    missing_relevant = [doc_id for doc_id, pos in relevant_positions.items() if pos is None]
    below_top5 = [doc_id for doc_id, pos in relevant_positions.items() if pos is not None and pos > 5]
    recall5 = recall_at_k(found20[:5], relevant)
    recall20 = recall_at_k(found20[:20], relevant)

    reasons = []
    if missing_relevant:
        reasons.append("relevant docs отсутствуют в top-100 candidate pool")
    if below_top5:
        reasons.append("relevant docs есть ниже top-5")
    if q["kind"] == "semantic":
        reasons.append("возможна сильная переформулировка запроса")
    if q["kind"] == "exact":
        reasons.append("dense-поиск может плохо ловить точные термины/коды")
    if q["kind"] == "multi":
        reasons.append("multi-запрос может требовать декомпозиции")

    if recall5 < 1.0:
        problem_rows.append({
            "kind": q["kind"],
            "query": q["query"],
            "relevant_docs": relevant,
            "found_top5": found20[:5],
            "found_top20": found20[:20],
            "recall@5": recall5,
            "recall@20": recall20,
            "relevant_positions_in_top100_docs": relevant_positions,
            "diagnosis": "; ".join(reasons),
        })

problem_queries = pd.DataFrame(problem_rows).sort_values(["recall@5", "recall@20", "kind"]).reset_index(drop=True)
print("Проблемных запросов с recall@5 < 1.0:", len(problem_queries))
problem_queries.head(15)

Проблемных запросов с recall@5 < 1.0: 17


,kind,query,relevant_docs,found_top5,found_top20,recall@5,recall@20,relevant_positions_in_top100_docs,diagnosis
0,semantic,"в приложении всё включено, а на телефон ничего...",[124],"[72, 66, 172, 117, 21]","[72, 66, 172, 117, 21, 30, 129, 41, 131, 166, ...",0.0,0.0,{124: 30},relevant docs есть ниже top-5; возможна сильна...
1,multi,заплатил за новинку отдельно — сколько она у м...,"[73, 140]","[116, 40, 141, 103, 3]","[116, 40, 141, 103, 3, 0, 24, 29, 21, 108, 73,...",0.0,1.0,"{73: 11, 140: 19}",relevant docs есть ниже top-5; multi-запрос мо...
2,semantic,"не пускает в аккаунт, пароль вылетел из головы",[55],"[9, 72, 117, 173, 166]","[9, 72, 117, 173, 166, 21, 140, 7, 41, 130, 97...",0.0,1.0,{55: 14},relevant docs есть ниже top-5; возможна сильна...
3,semantic,"боюсь, что ребёнок случайно купит фильм с моей...",[140],"[108, 73, 20, 116, 96]","[108, 73, 20, 116, 96, 173, 171, 47, 141, 140,...",0.0,1.0,{140: 10},relevant docs есть ниже top-5; возможна сильна...
4,semantic,"как посмотреть, за что с меня списали деньги в...",[106],"[5, 64, 116, 154, 173]","[5, 64, 116, 154, 173, 162, 115, 155, 122, 128...",0.0,1.0,{106: 12},relevant docs есть ниже top-5; возможна сильна...
5,semantic,"хочу убрать из ленты фильмы, которые уже смотрел",[97],"[46, 20, 81, 119, 157]","[46, 20, 81, 119, 157, 168, 112, 108, 7, 135, ...",0.0,1.0,{97: 11},relevant docs есть ниже top-5; возможна сильна...
6,semantic,"видят ли другие люди, что я поставил фильму лайк",[135],"[99, 157, 173, 134, 41]","[99, 157, 173, 134, 41, 96, 108, 24, 46, 81, 9...",0.0,1.0,{135: 13},relevant docs есть ниже top-5; возможна сильна...
7,multi,чем Стандарт отличается от Максимума и потеряю...,"[113, 128]","[155, 128, 64, 24, 180]","[155, 128, 64, 24, 180, 162, 116, 29, 32, 154,...",0.5,1.0,"{113: 15, 128: 2}",relevant docs есть ниже top-5; multi-запрос мо...
8,multi,сколько устройств могут смотреть одновременно ...,"[85, 113]","[85, 44, 166, 24, 110]","[85, 44, 166, 24, 110, 131, 99, 144, 107, 1, 3...",0.5,1.0,"{85: 1, 113: 14}",relevant docs есть ниже top-5; multi-запрос мо...
9,multi,за границей при запуске фильма появляется P-10...,"[20, 47]","[47, 90, 94, 51, 137]","[47, 90, 94, 51, 137, 49, 20, 138, 23, 178, 96...",0.5,1.0,"{20: 7, 47: 1}",relevant docs есть ниже top-5; multi-запрос мо...


### Вывод по этапу 3

Этап 3 пересчитан на выбранной стратегии `semantic_500_1` и том же `indexable_docs`. FAISS использует `IndexFlatIP`: эмбеддинги нормализованы, поэтому inner product эквивалентен cosine similarity; корпус малый, поэтому плоский CPU-индекс достаточен и не требует approximate search.

Baseline dense retrieval агрегирует найденные чанки в уникальные `doc_id`. Лучший baseline-вариант с overfetch `dense_chunk100_doc20` даёт `precision@5=0.203571`, `recall@5=0.785714`, `MRR=0.700836`, `recall@1=0.464286`, `recall@3=0.705357`, `recall@10=0.848214`, `recall@20=0.982143`. По типам запросов: `exact` достигает `recall@5=1.000000`, `multi` — `0.656250`, `semantic` — `0.740000`; основные потери остаются в multi-hop и разговорных переформулировках.

Проверен metadata filtering на категории `Подписка и оплата`: фильтр возвращает документы только нужной категории и демонстрирует, что category post-filter можно применять как диагностический или продуктовый ограничитель выдачи. Порог similarity подобран по распределению score, но в текущем baseline остаётся диагностическим: жёсткое применение threshold на малом eval-наборе может ухудшить recall, поэтому финальный retrieval в следующих пунктах должен улучшаться через candidate generation/reranking, а не через раннее отсечение кандидатов.


## Этап 4. Диагноз и улучшение retrieval

На этом шаге улучшаем candidate generation относительно dense-only baseline: добавляем структурированный `searchable_text`, BM25, dense+BM25 RRF и простые rule-based трансформации запросов. После диагностики высокого `recall@20` и недостаточного `recall@5` дополнительно проверяем cross-encoder reranker для перестановки top-100 dense chunk-кандидатов.


In [20]:
from rank_bm25 import BM25Okapi

STEP5_CHUNK_K = 100
STEP5_DOC_K = 20
RRF_K = 60

def build_searchable_text(title: str, category: str, text: str) -> str:
    """Создаёт структурированный текст только для retrieval-индексации.
    
        Args:
            title: Заголовок документа.
            category: Категория документа.
            text: Исходный текст чанка.
    
        Returns:
            Структурированный searchable text с добавленными заголовком и категорией.
    """
    parts = [
        f"\u0417\u0430\u0433\u043e\u043b\u043e\u0432\u043e\u043a: {title}",
        f"\u041a\u0430\u0442\u0435\u0433\u043e\u0440\u0438\u044f: {category}",
        f"\u0422\u0435\u043a\u0441\u0442: {text}",
    ]
    return "\n".join(parts)


def add_searchable_text(chunks):
    """Добавляет `searchable_text`, сохраняя исходный `page_content`.
    
        Args:
            chunks: Исходные словари чанков.
    
        Returns:
            Словари чанков с дополнительным полем `searchable_text`.
    """
    enhanced = []
    for chunk in chunks:
        item = dict(chunk)
        item["searchable_text"] = build_searchable_text(
            title=item["title"],
            category=item["category"],
            text=item["page_content"],
        )
        enhanced.append(item)
    return enhanced


enhanced_chunks = add_searchable_text(selected_chunks)
enhanced_embeddings = encode_passages([c["searchable_text"] for c in enhanced_chunks])
enhanced_index = build_faiss_index(enhanced_embeddings)

print("Улучшенных чанков:", len(enhanced_chunks))
print("Эмбеддинги улучшенных чанков:", enhanced_embeddings.shape)
print("Размер улучшенного индекса:", enhanced_index.ntotal)


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Улучшенных чанков: 452
Эмбеддинги улучшенных чанков: (452, 768)
Размер улучшенного индекса: 452


In [21]:
def dense_enhanced_chunk_search(query_text: str, chunk_k: int = STEP5_CHUNK_K):
    """Ищет в улучшенном dense-индексе по структурированному searchable text.
    
        Args:
            query_text: Пользовательский запрос или преобразованный текст запроса.
            chunk_k: Количество chunk-кандидатов, запрашиваемых у FAISS.
    
        Returns:
            Список chunk hits с rank, score и исходными метаданными.
    """
    q_vec = encode_queries([query_text])[0]
    k = min(chunk_k, len(enhanced_chunks))
    scores, idx = enhanced_index.search(q_vec.reshape(1, -1), k)
    hits = []
    for rank, (score, chunk_idx) in enumerate(zip(scores[0], idx[0]), start=1):
        if chunk_idx < 0:
            continue
        chunk = enhanced_chunks[int(chunk_idx)]
        hits.append({
            "rank": rank,
            "chunk_index": int(chunk_idx),
            "score": float(score),
            "doc_id": chunk["doc_id"],
            "title": chunk["title"],
            "category": chunk["category"],
            "chunk_id": chunk["chunk_id"],
            "page_content": chunk["page_content"],
        })
    return hits


def doc_hits_from_chunk_hits(chunk_hits, doc_k: int = STEP5_DOC_K):
    """Агрегирует chunk hits в ранжированные document hits.
    
        Args:
            chunk_hits: Результаты поиска уровня чанков.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            Ранжированные document hits с исходным текстом для сборки контекста.
    """
    return aggregate_chunks_to_docs(chunk_hits, doc_k=doc_k)


def dense_enhanced_doc_hits(query_text: str, chunk_k: int = STEP5_CHUNK_K, doc_k: int = STEP5_DOC_K):
    """Возвращает document hits из улучшенного dense retrieval.
    
        Args:
            query_text: Пользовательский запрос или преобразованный текст запроса.
            chunk_k: Количество chunk-кандидатов, запрашиваемых у FAISS.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            Ранжированные document hits, агрегированные из улучшенных dense chunk hits.
    """
    return doc_hits_from_chunk_hits(dense_enhanced_chunk_search(query_text, chunk_k=chunk_k), doc_k=doc_k)

In [22]:
def tokenize_for_bm25(text: str):
    """Токенизирует русский текст для BM25 простыми regex-правилами в нижнем регистре.
    
        Args:
            text: Входной текст документа или запроса.
    
        Returns:
            Список буквенно-цифровых токенов в нижнем регистре без односимвольных токенов.
    """
    return [tok for tok in re.findall(r"[a-z\u0430-\u044f\u04510-9]+", text.lower()) if len(tok) > 1]


bm25_corpus = [tokenize_for_bm25(c["searchable_text"]) for c in enhanced_chunks]
bm25_index = BM25Okapi(bm25_corpus)

def bm25_chunk_search(query_text: str, chunk_k: int = STEP5_CHUNK_K):
    """Ищет чанки через BM25 по улучшенному searchable text.
    
        Args:
            query_text: Пользовательский запрос или преобразованный текст запроса.
            chunk_k: Количество возвращаемых chunk-кандидатов.
    
        Returns:
            Список BM25 chunk hits с метаданными и исходным page content.
    """
    tokens = tokenize_for_bm25(query_text)
    scores = bm25_index.get_scores(tokens)
    if len(scores) == 0:
        return []
    top_idx = np.argsort(scores)[::-1][:min(chunk_k, len(scores))]
    hits = []
    for rank, chunk_idx in enumerate(top_idx, start=1):
        score = float(scores[int(chunk_idx)])
        chunk = enhanced_chunks[int(chunk_idx)]
        hits.append({
            "rank": rank,
            "chunk_index": int(chunk_idx),
            "score": score,
            "doc_id": chunk["doc_id"],
            "title": chunk["title"],
            "category": chunk["category"],
            "chunk_id": chunk["chunk_id"],
            "page_content": chunk["page_content"],
        })
    return hits


def bm25_doc_hits(query_text: str, chunk_k: int = STEP5_CHUNK_K, doc_k: int = STEP5_DOC_K):
    """Возвращает document hits из BM25 retrieval.
    
        Args:
            query_text: Пользовательский запрос или преобразованный текст запроса.
            chunk_k: Количество chunk-кандидатов, запрашиваемых у BM25.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            Ранжированные document hits, агрегированные из BM25 chunk hits.
    """
    return doc_hits_from_chunk_hits(bm25_chunk_search(query_text, chunk_k=chunk_k), doc_k=doc_k)

print("Размер корпуса BM25:", len(bm25_corpus))
print("Медианное число BM25-токенов на чанк:", int(np.median([len(x) for x in bm25_corpus])))

Размер корпуса BM25: 452
Медианное число BM25-токенов на чанк: 59


In [23]:
def reciprocal_rank_fusion(ranked_lists, k: int = RRF_K, doc_k: int = STEP5_DOC_K):
    """Объединяет несколько ранжированных списков документов через Reciprocal Rank Fusion.
    
        Args:
            ranked_lists: Последовательность ранжированных списков document hits.
            k: Сглаживающая константа RRF.
            doc_k: Количество возвращаемых объединённых документов.
    
        Returns:
            Ранжированные объединённые document hits с `rrf_score` и исходными метаданными.
    """
    fused = {}
    for ranked in ranked_lists:
        for rank, hit in enumerate(ranked, start=1):
            doc_id = hit["doc_id"]
            if doc_id not in fused:
                fused[doc_id] = dict(hit)
                fused[doc_id]["rrf_score"] = 0.0
                fused[doc_id]["source_count"] = 0
            fused[doc_id]["rrf_score"] += 1.0 / (k + rank)
            fused[doc_id]["source_count"] += 1
    rows = sorted(
        fused.values(),
        key=lambda row: (-row["rrf_score"], -row.get("source_count", 0), row.get("best_chunk_rank", row.get("rank", 9999))),
    )[:doc_k]
    for rank, row in enumerate(rows, start=1):
        row["doc_rank"] = rank
    return rows


def hybrid_rrf_doc_hits(query_text: str, chunk_k: int = STEP5_CHUNK_K, doc_k: int = STEP5_DOC_K):
    """Возвращает document hits, объединённые из улучшенного dense и BM25 retrieval.
    
        Args:
            query_text: Пользовательский запрос или преобразованный текст запроса.
            chunk_k: Количество chunk-кандидатов для каждого retriever.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            Ранжированные document hits после RRF-объединения dense+BM25.
    """
    dense_docs = dense_enhanced_doc_hits(query_text, chunk_k=chunk_k, doc_k=doc_k)
    bm25_docs = bm25_doc_hits(query_text, chunk_k=chunk_k, doc_k=doc_k)
    return reciprocal_rank_fusion([dense_docs, bm25_docs], doc_k=doc_k)

In [24]:
QUERY_EXPANSION_RULES = {
    "\u043e\u043f\u043b\u0430\u0442": "\u043e\u043f\u043b\u0430\u0442\u0430 \u043f\u043b\u0430\u0442\u0435\u0436 \u043a\u0430\u0440\u0442\u0430 \u0441\u043f\u0438\u0441\u0430\u043d\u0438\u0435 \u0447\u0435\u043a \u043a\u0432\u0438\u0442\u0430\u043d\u0446\u0438\u044f \u043f\u043e\u0434\u043f\u0438\u0441\u043a\u0430",
    "\u043a\u0430\u0440\u0442": "\u0431\u0430\u043d\u043a\u043e\u0432\u0441\u043a\u0430\u044f \u043a\u0430\u0440\u0442\u0430 \u043e\u043f\u043b\u0430\u0442\u0430 \u043f\u0440\u0438\u0432\u044f\u0437\u0430\u0442\u044c \u043e\u0442\u0432\u044f\u0437\u0430\u0442\u044c \u0441\u043f\u043e\u0441\u043e\u0431 \u043e\u043f\u043b\u0430\u0442\u044b",
    "\u043f\u043e\u0434\u043f\u0438\u0441": "\u043f\u043e\u0434\u043f\u0438\u0441\u043a\u0430 \u0442\u0430\u0440\u0438\u0444 \u043f\u0440\u043e\u0434\u043b\u0435\u043d\u0438\u0435 \u0430\u0432\u0442\u043e\u043f\u0440\u043e\u0434\u043b\u0435\u043d\u0438\u0435 \u043e\u0442\u043a\u043b\u044e\u0447\u0438\u0442\u044c",
    "\u043e\u0448\u0438\u0431": "\u043e\u0448\u0438\u0431\u043a\u0430 \u043a\u043e\u0434 \u0432\u043e\u0441\u043f\u0440\u043e\u0438\u0437\u0432\u0435\u0434\u0435\u043d\u0438\u0435 \u043f\u043b\u0435\u0435\u0440 \u043f\u0440\u043e\u0431\u043b\u0435\u043c\u0430",
    "\u0443\u0441\u0442\u0440\u043e\u0439": "\u0443\u0441\u0442\u0440\u043e\u0439\u0441\u0442\u0432\u043e \u0442\u0435\u043b\u0435\u0432\u0438\u0437\u043e\u0440 \u0441\u043c\u0430\u0440\u0442\u0444\u043e\u043d \u043f\u043b\u0430\u043d\u0448\u0435\u0442 \u043f\u0440\u0438\u0441\u0442\u0430\u0432\u043a\u0430 \u043f\u043e\u0434\u043a\u043b\u044e\u0447\u0438\u0442\u044c",
    "\u043f\u0440\u043e\u0444\u0438\u043b": "\u043f\u0440\u043e\u0444\u0438\u043b\u044c \u0434\u0435\u0442\u0441\u043a\u0438\u0439 \u043f\u0440\u043e\u0444\u0438\u043b\u044c \u0430\u043a\u043a\u0430\u0443\u043d\u0442 \u043f\u043e\u043b\u044c\u0437\u043e\u0432\u0430\u0442\u0435\u043b\u044c",
    "\u0430\u043a\u043a\u0430\u0443\u043d\u0442": "\u0430\u043a\u043a\u0430\u0443\u043d\u0442 \u0432\u0445\u043e\u0434 \u043f\u0430\u0440\u043e\u043b\u044c \u043f\u043e\u0447\u0442\u0430 \u0442\u0435\u043b\u0435\u0444\u043e\u043d \u043f\u0440\u043e\u0444\u0438\u043b\u044c",
}

def expand_query_rule_based(query_text: str):
    """Расширяет semantic-запрос компактными доменными синонимами.
    
        Args:
            query_text: Исходный пользовательский запрос.
    
        Returns:
            Расширенная строка запроса. Если правило не найдено, возвращается исходный запрос.
    """
    lowered = query_text.lower()
    additions = []
    for marker, expansion in QUERY_EXPANSION_RULES.items():
        if marker in lowered:
            additions.append(expansion)
    if not additions:
        return query_text
    return query_text + " " + " ".join(dict.fromkeys(" ".join(additions).split()))


def decompose_query_rule_based(query_text: str):
    """Разбивает multi-intent запросы на простые подзапросы.
    
        Args:
            query_text: Исходный пользовательский запрос.
    
        Returns:
            Список с исходным запросом и подзапросами, полученными по правилам.
    """
    split_pattern = r"\s+(?:или|а\s+также|плюс|и)\s+|[,;]"
    parts = [p.strip(" ,.;:!?()") for p in re.split(split_pattern, query_text, flags=re.IGNORECASE)]
    parts = [p for p in parts if len(p) >= 8]
    queries_out = [query_text]
    for part in parts:
        if part.lower() != query_text.lower() and part not in queries_out:
            queries_out.append(part)
    return queries_out


def hybrid_rrf_expanded_doc_hits(query_text: str, kind: str, chunk_k: int = STEP5_CHUNK_K, doc_k: int = STEP5_DOC_K):
    """Запускает hybrid retrieval с expansion для semantic-запросов.
    
        Args:
            query_text: Исходный пользовательский запрос.
            kind: Тип запроса из evaluation-набора.
            chunk_k: Количество chunk-кандидатов для каждого retriever.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            Ранжированные document hits после необязательного expansion и RRF.
    """
    variants = [query_text]
    if kind == "semantic":
        expanded = expand_query_rule_based(query_text)
        if expanded != query_text:
            variants.append(expanded)
    ranked_lists = [hybrid_rrf_doc_hits(v, chunk_k=chunk_k, doc_k=doc_k) for v in variants]
    return reciprocal_rank_fusion(ranked_lists, doc_k=doc_k)


def hybrid_rrf_expanded_decomposed_doc_hits(query_text: str, kind: str, chunk_k: int = STEP5_CHUNK_K, doc_k: int = STEP5_DOC_K):
    """Запускает hybrid retrieval с semantic expansion и multi-query decomposition.
    
        Args:
            query_text: Исходный пользовательский запрос.
            kind: Тип запроса из evaluation-набора.
            chunk_k: Количество chunk-кандидатов для каждого retriever.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            Ранжированные document hits после необязательных преобразований и RRF.
    """
    variants = [query_text]
    if kind == "semantic":
        expanded = expand_query_rule_based(query_text)
        if expanded != query_text:
            variants.append(expanded)
    if kind == "multi":
        variants.extend(decompose_query_rule_based(query_text)[1:])
    variants = list(dict.fromkeys(variants))
    ranked_lists = [hybrid_rrf_doc_hits(v, chunk_k=chunk_k, doc_k=doc_k) for v in variants]
    return reciprocal_rank_fusion(ranked_lists, doc_k=doc_k)


In [25]:
def evaluate_retriever_variant(variant: str, retrieve_docs_fn, doc_k: int = STEP5_DOC_K):
    """Оценивает document-level retriever на evaluation-запросах без gap.
    
        Args:
            variant: Человекочитаемое название варианта.
            retrieve_docs_fn: Функция, принимающая один словарь запроса и возвращающая document hits.
            doc_k: Количество top-документов, используемых для метрик.
    
        Returns:
            `pandas.DataFrame` с retrieval-метриками по каждому запросу.
    """
    rows = []
    for q in eval_queries:
        doc_hits = retrieve_docs_fn(q)[:doc_k]
        found_docs = [row["doc_id"] for row in doc_hits]
        row = {
            "variant": variant,
            "kind": q["kind"],
            "query": q["query"],
            "relevant_docs": q["relevant_docs"],
            "found_doc_ids": found_docs,
            "precision@5": precision_at_k(found_docs[:5], q["relevant_docs"]),
            "recall@5": recall_at_k(found_docs[:5], q["relevant_docs"]),
            "MRR": mrr(found_docs, q["relevant_docs"]),
        }
        for k in RECALL_KS:
            row[f"recall@{k}"] = recall_at_k(found_docs[:k], q["relevant_docs"])
        rows.append(row)
    return pd.DataFrame(rows)


step5_results = [
    evaluate_retriever_variant(
        "dense_enhanced_chunk100_doc20",
        lambda q: dense_enhanced_doc_hits(q["query"], chunk_k=STEP5_CHUNK_K, doc_k=STEP5_DOC_K),
    ),
    evaluate_retriever_variant(
        "bm25_chunk100_doc20",
        lambda q: bm25_doc_hits(q["query"], chunk_k=STEP5_CHUNK_K, doc_k=STEP5_DOC_K),
    ),
    evaluate_retriever_variant(
        "hybrid_dense_bm25_rrf_doc20",
        lambda q: hybrid_rrf_doc_hits(q["query"], chunk_k=STEP5_CHUNK_K, doc_k=STEP5_DOC_K),
    ),
    evaluate_retriever_variant(
        "hybrid_rrf_expanded_semantic_doc20",
        lambda q: hybrid_rrf_expanded_doc_hits(q["query"], q["kind"], chunk_k=STEP5_CHUNK_K, doc_k=STEP5_DOC_K),
    ),
    evaluate_retriever_variant(
        "hybrid_rrf_expanded_decomposed_doc20",
        lambda q: hybrid_rrf_expanded_decomposed_doc_hits(q["query"], q["kind"], chunk_k=STEP5_CHUNK_K, doc_k=STEP5_DOC_K),
    ),
]

step5_diagnostics = pd.concat(step5_results, ignore_index=True)
step5_summary = step5_diagnostics.groupby("variant")[[
    "precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"
]].mean().reset_index().sort_values(["recall@5", "MRR", "recall@20"], ascending=False)
step5_summary

,variant,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
1,dense_enhanced_chunk100_doc20,0.192857,0.758929,0.736004,0.526786,0.696429,0.875000,0.982143
2,hybrid_dense_bm25_rrf_doc20,0.189286,0.723214,0.671523,0.473214,0.660714,0.839286,0.937500
4,hybrid_rrf_expanded_semantic_doc20,0.182143,0.696429,0.671521,0.473214,0.633929,0.875000,0.937500
3,hybrid_rrf_expanded_decomposed_doc20,0.167857,0.660714,0.642921,0.455357,0.580357,0.875000,0.937500
0,bm25_chunk100_doc20,0.150000,0.580357,0.519779,0.321429,0.517857,0.687500,0.785714


### Cross-encoder reranker для лучшего candidate pool

Если диагностика показывает, что релевантные документы часто есть в расширенном candidate pool, но не попадают в top-5, проверяется reranker: retrieval формирует расширенный список chunk-кандидатов, cross-encoder заново оценивает пары `запрос → чанк`, а затем чанки агрегируются в уникальные документы.

Цена улучшения должна быть названа явно: дополнительная модель, рост задержки и усложнение pipeline.


In [26]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
RERANKER_CANDIDATE_CHUNK_K = STEP5_CHUNK_K
RERANKER_DOC_K = STEP5_DOC_K

reranker = CrossEncoder(RERANKER_MODEL_NAME)


def format_reranker_passage(hit) -> str:
    """Собирает текст чанка с метаданными для cross-encoder reranker.
    
        Args:
            hit: Chunk hit с заголовком, категорией и исходным page content.
    
        Returns:
            Строка passage для оценки пары `запрос -> чанк`.
    """
    return "\n".join([
        f"Заголовок: {hit['title']}",
        f"Категория: {hit['category']}",
        f"Текст: {hit['page_content']}",
    ])


def rerank_chunk_hits_to_doc_hits(query_text: str, chunk_hits, doc_k: int = RERANKER_DOC_K):
    """Переранжирует dense chunk-кандидаты cross-encoder моделью и агрегирует их в документы.
    
        Args:
            query_text: Исходный пользовательский запрос.
            chunk_hits: Chunk-кандидаты из dense retrieval.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            Ранжированные document hits после reranking по лучшему reranker-score чанка.
    """
    if not chunk_hits:
        return []
    pairs = [(query_text, format_reranker_passage(hit)) for hit in chunk_hits]
    rerank_scores = reranker.predict(pairs, show_progress_bar=False)
    reranked_chunk_hits = []
    for hit, score in zip(chunk_hits, rerank_scores):
        row = dict(hit)
        row["dense_score"] = row["score"]
        row["score"] = float(score)
        row["reranker_score"] = float(score)
        reranked_chunk_hits.append(row)
    reranked_chunk_hits = sorted(
        reranked_chunk_hits,
        key=lambda row: (-row["reranker_score"], row["rank"]),
    )
    for rank, row in enumerate(reranked_chunk_hits, start=1):
        row["rank"] = rank
    return aggregate_chunks_to_docs(reranked_chunk_hits, doc_k=doc_k)


def dense_enhanced_reranked_doc_hits(query_text: str, chunk_k: int = RERANKER_CANDIDATE_CHUNK_K, doc_k: int = RERANKER_DOC_K):
    """Возвращает document hits после dense candidate generation и cross-encoder reranking.
    
        Args:
            query_text: Исходный пользовательский запрос.
            chunk_k: Количество dense chunk-кандидатов для reranker.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            Ранжированные document hits после reranking.
    """
    chunk_hits = dense_enhanced_chunk_search(query_text, chunk_k=chunk_k)
    return rerank_chunk_hits_to_doc_hits(query_text, chunk_hits, doc_k=doc_k)


reranker_diagnostics = evaluate_retriever_variant(
    "dense_enhanced_reranked_chunk100_doc20",
    lambda q: dense_enhanced_reranked_doc_hits(q["query"], chunk_k=RERANKER_CANDIDATE_CHUNK_K, doc_k=RERANKER_DOC_K),
)

step5_diagnostics = pd.concat([step5_diagnostics, reranker_diagnostics], ignore_index=True)
step5_summary = step5_diagnostics.groupby("variant")[[
    "precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"
]].mean().reset_index().sort_values(["recall@5", "MRR", "recall@20"], ascending=False)

reranker_comparison = step5_summary[
    step5_summary["variant"].isin(["dense_enhanced_chunk100_doc20", "dense_enhanced_reranked_chunk100_doc20"])
].copy()
reranker_comparison


,variant,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
2,dense_enhanced_reranked_chunk100_doc20,0.239286,0.901786,0.802381,0.580357,0.767857,0.946429,0.973214
1,dense_enhanced_chunk100_doc20,0.192857,0.758929,0.736004,0.526786,0.696429,0.875000,0.982143


In [27]:
step5_by_kind = step5_diagnostics.groupby(["variant", "kind"])[[
    "precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"
]].mean().reset_index()
step5_by_kind.sort_values(["variant", "kind"])

,variant,kind,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
0,bm25_chunk100_doc20,exact,0.200000,0.966667,0.888889,0.766667,0.966667,0.966667,1.00000
1,bm25_chunk100_doc20,multi,0.212500,0.531250,0.603676,0.218750,0.468750,0.656250,0.78125
2,bm25_chunk100_doc20,semantic,0.080000,0.380000,0.244618,0.120000,0.280000,0.540000,0.66000
3,dense_enhanced_chunk100_doc20,exact,0.213333,1.000000,0.966667,0.900000,0.966667,1.000000,1.00000
4,dense_enhanced_chunk100_doc20,multi,0.250000,0.625000,0.819444,0.375000,0.562500,0.843750,1.00000
5,dense_enhanced_chunk100_doc20,semantic,0.144000,0.700000,0.544205,0.400000,0.620000,0.820000,0.96000
6,dense_enhanced_reranked_chunk100_doc20,exact,0.213333,1.000000,1.000000,0.966667,1.000000,1.000000,1.00000
7,dense_enhanced_reranked_chunk100_doc20,multi,0.350000,0.875000,0.927083,0.437500,0.718750,0.906250,0.96875
8,dense_enhanced_reranked_chunk100_doc20,semantic,0.184000,0.860000,0.604000,0.440000,0.660000,0.940000,0.96000
9,hybrid_dense_bm25_rrf_doc20,exact,0.213333,1.000000,0.966667,0.900000,0.966667,1.000000,1.00000


In [28]:
baseline_best_name = "dense_chunk100_doc20"
baseline_best_metrics = baseline_summary[baseline_summary["variant"] == baseline_best_name].copy()
if baseline_best_metrics.empty:
    baseline_best_metrics = baseline_summary.head(1).copy()
    baseline_best_name = baseline_best_metrics.iloc[0]["variant"]

best_step5_variant = step5_summary.iloc[0]["variant"]
best_step5_metrics = step5_summary[step5_summary["variant"] == best_step5_variant].copy()

comparison_before_after = pd.concat([
    baseline_best_metrics.assign(stage="before_step5"),
    best_step5_metrics.assign(stage="after_step5"),
], ignore_index=True)[[
    "stage", "variant", "precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"
]]

metric_cols = ["precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"]
improvements = {
    f"delta_{col}": float(best_step5_metrics.iloc[0][col] - baseline_best_metrics.iloc[0][col])
    for col in metric_cols
}

print("Базовый вариант:", baseline_best_name)
print("Лучший вариант шага 5:", best_step5_variant)
print("Изменения метрик:", {k: round(v, 6) for k, v in improvements.items()})
comparison_before_after

Базовый вариант: dense_chunk100_doc20
Лучший вариант шага 5: dense_enhanced_reranked_chunk100_doc20
Изменения метрик: {'delta_precision@5': 0.035714, 'delta_recall@5': 0.116071, 'delta_MRR': 0.101545, 'delta_recall@1': 0.116071, 'delta_recall@3': 0.0625, 'delta_recall@10': 0.098214, 'delta_recall@20': -0.008929}


,stage,variant,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
0,before_step5,dense_chunk100_doc20,0.203571,0.785714,0.700836,0.464286,0.705357,0.848214,0.982143
1,after_step5,dense_enhanced_reranked_chunk100_doc20,0.239286,0.901786,0.802381,0.580357,0.767857,0.946429,0.973214


In [29]:
best_step5_rows = step5_diagnostics[step5_diagnostics["variant"] == best_step5_variant].copy()
remaining_problem_queries = best_step5_rows[best_step5_rows["recall@5"] < 1.0].sort_values([
    "recall@5", "recall@20", "kind"
]).reset_index(drop=True)

best_recall5 = float(best_step5_metrics.iloc[0]["recall@5"])
best_recall20 = float(best_step5_metrics.iloc[0]["recall@20"])
reranker_was_evaluated = "dense_enhanced_reranked_chunk100_doc20" in set(step5_diagnostics["variant"])
if best_recall5 >= 0.85:
    next_retrieval_step = "move to answer generation and LLM judge"
elif reranker_was_evaluated:
    next_retrieval_step = "document reranker limitation and move to answer generation and LLM judge"
elif best_recall20 >= 0.90:
    next_retrieval_step = "connect reranker: candidate pool is broad enough, but top-5 ranking is weak"
else:
    next_retrieval_step = "continue candidate generation: recall@20 is still below 0.90"

print("Проблемных запросов для лучшего варианта шага 5 с recall@5 < 1.0:", len(remaining_problem_queries))
print("Следующее решение по retrieval:", next_retrieval_step)
remaining_problem_queries.head(15)


Проблемных запросов для лучшего варианта шага 5 с recall@5 < 1.0: 8
Следующее решение по retrieval: move to answer generation and LLM judge


,variant,kind,query,relevant_docs,found_doc_ids,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
0,dense_enhanced_reranked_chunk100_doc20,semantic,"в приложении всё включено, а на телефон ничего...",[124],"[172, 66, 1, 150, 72, 31, 159, 117, 144, 166, ...",0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1,dense_enhanced_reranked_chunk100_doc20,semantic,скачанная серия перестала открываться без инте...,[49],"[109, 144, 177, 51, 99, 49, 15, 101, 107, 178,...",0.0,0.0,0.166667,0.0,0.0,1.0,1.0
2,dense_enhanced_reranked_chunk100_doc20,semantic,"как посмотреть, за что с меня списали деньги в...",[106],"[5, 155, 61, 115, 128, 106, 162, 64, 140, 180,...",0.0,0.0,0.166667,0.0,0.0,1.0,1.0
3,dense_enhanced_reranked_chunk100_doc20,multi,"у меня телевизор AM-9, приложение не ставится ...","[139, 175]","[139, 66, 112, 172, 176, 54, 178, 72, 129, 111...",0.2,0.5,1.000000,0.5,0.5,0.5,0.5
4,dense_enhanced_reranked_chunk100_doc20,multi,сколько устройств могут смотреть одновременно ...,"[85, 113]","[85, 99, 155, 166, 24, 128, 144, 131, 44, 62, ...",0.2,0.5,1.000000,0.5,0.5,0.5,1.0
5,dense_enhanced_reranked_chunk100_doc20,multi,"скачал сериал в дорогу, а место на телефоне ко...","[144, 178]","[144, 177, 109, 51, 137, 178, 44, 111, 49, 20,...",0.2,0.5,1.000000,0.5,0.5,1.0,1.0
6,dense_enhanced_reranked_chunk100_doc20,multi,собираюсь в отпуск за рубеж и хочу взять кино ...,"[20, 144]","[20, 99, 73, 134, 46, 155, 91, 47, 119, 109, 1...",0.2,0.5,1.000000,0.5,0.5,0.5,1.0
7,dense_enhanced_reranked_chunk100_doc20,semantic,"как сделать, чтобы деньги сами не снимались ка...","[61, 64]","[155, 5, 128, 154, 61, 106, 7, 35, 115, 140, 1...",0.2,0.5,0.200000,0.0,0.0,0.5,1.0


### Дополнительный аудит чанкинга, candidate pool и reranking

Дополнительные retrieval-эксперименты допустимы только как часть диагноза этапа 4. Все варианты нужно сравнивать на одном и том же наборе non-gap запросов и одинаковых метриках. Финальные выводы должны ссылаться только на числа, которые присутствуют в outputs notebook.

Если query expansion, coverage rules или prompt-эвристики подбирались на тех же 56 non-gap запросах, это нужно честно назвать настройкой на диагностическом наборе, а не независимой оценкой обобщения.


In [30]:
chunking_experiment_summary = summary.rename(columns={"strategy": "variant"}).copy()
chunking_experiment_summary["family"] = chunking_experiment_summary["variant"].str.extract(r"^([a-z_]+)")[0]
chunking_experiment_summary = chunking_experiment_summary.sort_values(
    ["recall@5", "MRR", "recall@20"],
    ascending=False,
).reset_index(drop=True)

best_chunking_variant = chunking_experiment_summary.iloc[0]["variant"]
best_chunking_metrics = chunking_experiment_summary.iloc[[0]].copy()

step5_reference_variants = step5_summary[
    step5_summary["variant"].isin([
        "dense_enhanced_chunk100_doc20",
        "dense_enhanced_reranked_chunk100_doc20",
    ])
].copy()
step5_reference_variants["family"] = "step5_reference"

chunking_vs_step5 = pd.concat([
    chunking_experiment_summary.head(10),
    step5_reference_variants,
], ignore_index=True, sort=False)[[
    "family", "variant", "precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"
]].sort_values(["recall@5", "MRR", "recall@20"], ascending=False)

print("Лучшая стратегия после аудита чанкинга:", best_chunking_variant)
print("Лучший вариант Step 5 до аудита чанкинга:", best_step5_variant)
chunking_vs_step5


Лучшая стратегия после аудита чанкинга: semantic_500_1
Лучший вариант Step 5 до аудита чанкинга: dense_enhanced_reranked_chunk100_doc20


,family,variant,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
10,step5_reference,dense_enhanced_reranked_chunk100_doc20,0.239286,0.901786,0.802381,0.580357,0.767857,0.946429,0.973214
0,semantic_,semantic_500_1,0.203571,0.785714,0.699463,0.464286,0.705357,0.839286,0.901786
1,sentence_,sentence_300_1,0.217857,0.785714,0.653323,0.437500,0.687500,0.848214,0.848214
11,step5_reference,dense_enhanced_chunk100_doc20,0.192857,0.758929,0.736004,0.526786,0.696429,0.875000,0.982143
2,sentence_,sentence_420_1,0.196429,0.750000,0.694118,0.491071,0.660714,0.875000,0.875000
3,semantic_,semantic_700_1,0.196429,0.750000,0.677169,0.464286,0.651786,0.883929,0.928571
4,sentence_,sentence_620_1,0.189286,0.714286,0.654500,0.446429,0.616071,0.919643,0.937500
5,full_doc,full_doc,0.185714,0.705357,0.564718,0.312500,0.607143,0.919643,0.982143
6,recursive_,recursive_350_70,0.182143,0.687500,0.586345,0.366071,0.535714,0.910714,0.910714
7,recursive_,recursive_250_50,0.178571,0.669643,0.665676,0.473214,0.598214,0.821429,0.875000


### Вывод по этапу 4

Этап 4 должен завершаться честным сравнением baseline и improved retrieval на одинаковой процедуре оценки. Итоговые `recall@5`, `MRR`, `recall@K` и breakdown по `exact`, `semantic`, `multi` должны относиться именно к финальной выбранной схеме, а не к промежуточному эксперименту.

После ревью прежние числовые выводы по улучшенному retrieval считаются предварительными: их нужно пересчитать после фильтрации корпуса и выбора финального pipeline. Отдельно нужно указать цену reranker и ограничение, что эвристики могли быть настроены на том же диагностическом наборе.


## Этап 5. Генерация ответов

На этом шаге поверх лучшей retrieval-схемы Step 5 собираем полный RAG pipeline: вопрос → retrieval → контекст → prompt → ответ. Финальная схема использует `dense_enhanced_reranked_chunk100_doc20`, то есть dense candidate generation по улучшенному searchable text и cross-encoder reranking. Поверх ранжирования добавлен штраф архивным/устаревшим статьям, чтобы актуальные документы попадали в контекст выше архивных. Генератор и предварительный LLM-судья использует локальный OpenAI-compatible endpoint. Если локальный endpoint недоступен, ячейки не падают: они показывают retrieval/context preview и фиксируют, что live-генерацию нужно выполнить после запуска локальной модели.


In [31]:
from openai import OpenAI
import urllib.request
import time

LOCAL_LLM_BASE_URL = "http://localhost:8080/v1"
LOCAL_LLM_MODELS_URL = "http://localhost:8080/v1/models"
LOCAL_LLM_MODEL = "unsloth/Qwen3.5-4B-GGUF:Q4_K_M"
GENERATION_TEMPERATURE = 0
GENERATION_MAX_TOKENS = 512
JUDGE_MAX_TOKENS = 192
JUDGE_RETRIES = 2
NO_THINK_EXTRA_BODY = {"chat_template_kwargs": {"enable_thinking": False}}
REFUSAL_PHRASE = "В базе знаний нет информации для ответа на этот вопрос."
FINAL_RETRIEVAL_VARIANT = "hybrid_dense_bm25_rrf_expanded_reranked_archive_penalty_coverage"
FINAL_BASE_RETRIEVAL_VARIANT = "dense_enhanced_reranked_chunk100_doc20"
FINAL_CANDIDATE_CHUNK_K = max(RERANKER_CANDIDATE_CHUNK_K, 150)
FINAL_DOC_POOL_K = max(RERANKER_DOC_K, 30)
FINAL_DOC_K = 5
MAX_CONTEXT_CHARS = 6500
GENERATION_CONTEXT_MAX_CHARS = 3200
GENERATION_CONTEXT_RETRY_CHARS = [3200, 2400, 1600, 1000, 700, 300]
JUDGE_CONTEXT_MAX_CHARS = 2400
JUDGE_CONTEXT_RETRY_CHARS = [2400, 1800, 1200, 800, 500, 300]
PREVIEW_LLM_ERROR_PREFIX = "LLM error"


def local_llm_is_available(timeout: float = 2.0) -> bool:
    """Проверяет доступность локального OpenAI-compatible endpoint.

    Args:
        timeout: Таймаут HTTP-запроса в секундах.

    Returns:
        `True`, если endpoint `/v1/models` отвечает без ошибки, иначе `False`.
    """
    try:
        with urllib.request.urlopen(LOCAL_LLM_MODELS_URL, timeout=timeout) as response:
            return 200 <= response.status < 300
    except Exception:
        return False


def make_llm_client(timeout: float = 120.0) -> OpenAI:
    """Создаёт OpenAI-compatible клиент для локальной LLM.

    Args:
        timeout: Клиентский timeout для generation/judge вызовов.

    Returns:
        Клиент `OpenAI`, настроенный на локальный endpoint.
    """
    return OpenAI(base_url=LOCAL_LLM_BASE_URL, api_key="not-needed", timeout=timeout)


def extract_chat_message_text(message) -> str:
    """Возвращает текст OpenAI-compatible chat message с учётом Qwen reasoning_content.

    Args:
        message: `completion.choices[0].message` из OpenAI-compatible API.

    Returns:
        Непустой текст из `content` или финальная часть `reasoning_content`, если сервер
        неожиданно вернул ответ в поле reasoning.
    """
    content = (getattr(message, "content", None) or "").strip()
    if content:
        return content
    reasoning = (getattr(message, "reasoning_content", None) or "").strip()
    if "</think>" in reasoning:
        return reasoning.split("</think>", 1)[1].strip()
    return reasoning


llm_available = local_llm_is_available()
print("Локальный LLM endpoint доступен:", llm_available)
print("Endpoint:", LOCAL_LLM_BASE_URL)
print("Модель:", LOCAL_LLM_MODEL)


Локальный LLM endpoint доступен: True
Endpoint: http://localhost:8080/v1
Модель: unsloth/Qwen3.5-4B-GGUF:Q4_K_M


In [32]:
def is_archive_hit(hit: dict) -> bool:
    """Определяет, относится ли найденный документ к архивным/устаревшим.

    Args:
        hit: Document hit с полями `title` и `page_content`.

    Returns:
        `True`, если документ выглядит архивным или устаревшим.
    """
    title = str(hit.get("title", "")).lower()
    text = str(hit.get("page_content", "")).lower()
    return (
        "архив" in title
        or "(архив)" in title
        or ("устарел" in text and "архив" in text)
        or "не ориентируйтесь на эту страницу" in text
    )


def apply_archive_penalty(doc_hits):
    """Понижает архивные статьи после основного ранжирования.

    Args:
        doc_hits: Ранжированные document hits.

    Returns:
        Список document hits, где актуальные статьи идут перед архивными при сохранении
        относительного порядка внутри каждой группы.
    """
    rows = []
    for idx, hit in enumerate(doc_hits, start=1):
        row = dict(hit)
        base_rank = int(row.get("doc_rank", row.get("rank", idx)))
        row["base_doc_rank"] = base_rank
        row["is_archive"] = is_archive_hit(row)
        row["archive_penalty"] = 1 if row["is_archive"] else 0
        row.setdefault("coverage_reason", "base")
        rows.append(row)
    rows = sorted(rows, key=lambda row: (row["archive_penalty"], row["base_doc_rank"]))
    for rank, row in enumerate(rows, start=1):
        row["doc_rank"] = rank
    return rows


def tokenize_for_bm25(text: str) -> list[str]:
    """Токенизирует русский текст для BM25 без внешних словарей.

    Args:
        text: Исходный текст.

    Returns:
        Список нормализованных токенов.
    """
    return re.findall(r"[a-zа-яё0-9]+", text.lower().replace("ё", "е"))


bm25_corpus_texts = [chunk["searchable_text"] for chunk in enhanced_chunks]
bm25_index = BM25Okapi([tokenize_for_bm25(text) for text in bm25_corpus_texts])


def bm25_enhanced_chunk_search(query_text: str, chunk_k: int = FINAL_CANDIDATE_CHUNK_K):
    """Ищет chunk-кандидаты BM25 по тому же searchable text, что и dense-index.

    Args:
        query_text: Пользовательский запрос или его честное расширение.
        chunk_k: Количество chunk-кандидатов.

    Returns:
        Список chunk hits с BM25 score.
    """
    tokens = tokenize_for_bm25(query_text)
    scores = bm25_index.get_scores(tokens)
    top_idx = sorted(range(len(scores)), key=lambda idx: float(scores[idx]), reverse=True)[:min(chunk_k, len(scores))]
    hits = []
    for rank, chunk_idx in enumerate(top_idx, start=1):
        chunk = enhanced_chunks[int(chunk_idx)]
        hits.append({
            "rank": rank,
            "chunk_index": int(chunk_idx),
            "score": float(scores[chunk_idx]),
            "doc_id": chunk["doc_id"],
            "title": chunk["title"],
            "category": chunk["category"],
            "chunk_id": chunk["chunk_id"],
            "page_content": chunk["page_content"],
            "retrieval_source": "bm25",
        })
    return hits


def expand_query_for_retrieval(query_text: str) -> list[str]:
    """Создаёт честные варианты запроса без привязки к `doc_id`.

    Args:
        query_text: Пользовательский вопрос.

    Returns:
        Небольшой список вариантов для dense/BM25 candidate generation.
    """
    normalized = query_text.strip()
    lower = normalized.lower().replace("ё", "е")
    variants = [normalized]

    if any(marker in lower for marker in ["за что", "прошл", "истор", "чек", "платеж", "платежи", "платёж", "платежей"]) and any(marker in lower for marker in ["спис", "деньги", "оплат", "плат"]):
        variants.append(
            f"{normalized}. история платежей раздел Платежи дата сумма способ оплаты каждого списания электронный чек"
        )
        variants.append("история платежей раздел Платежи дата сумма способ оплаты списания электронный чек")

    if any(marker in lower for marker in ["уже смотр", "смотрел", "смотрела", "просмотренн", "из ленты", "ленты", "продолжить просмотр"]) and any(marker in lower for marker in ["убрать", "удал", "скрыть", "не показыв", "ленты", "рекомендац"]):
        variants.append(
            f"{normalized}. история просмотров настройки профиля удалить запись убрать из истории продолжить просмотр рекомендации"
        )
        variants.append("история просмотров настройки профиля удалить запись убрать из истории продолжить просмотр рекомендации")

    if any(marker in lower for marker in ["одновременно", "одноврем", "сколько устройств", "сколько экран", "смотреть одновременно"]) and any(marker in lower for marker in ["превыш", "лимит", "что будет", "не хватает", "ошибка"]):
        variants.append(
            f"{normalized}. ошибка P-102 превышен лимит одновременных просмотров число экранов лимит тарифа"
        )
        variants.append("ошибка P-102 превышен лимит одновременных просмотров лимит тарифа сколько экранов")

    if any(marker in lower for marker in ["место", "памят", "свободн", "хранилищ", "storage"]) and any(marker in lower for marker in ["скач", "загруз", "офлайн", "сериал", "фильм", "телефон"]):
        variants.append(
            f"{normalized}. ошибка P-109 недостаточно места свободной памяти на устройстве загрузка сорвалась освободить память удалить старые загрузки"
        )
        variants.append("ошибка P-109 недостаточно места свободной памяти на устройстве загрузка сорвалась удалить старые загрузки")

    if any(marker in lower for marker in ["отпуск", "дорог", "поездк", "за рубеж", "за границ", "с собой", "взять кино"]) and any(marker in lower for marker in ["кино", "фильм", "сериал", "смотреть", "с собой", "скач", "офлайн"]):
        variants.append(
            f"{normalized}. скачать фильмы и сериалы офлайн просмотр в дороге без интернета загрузки на телефоне и планшете ограничения до 20 загрузок"
        )
        variants.append("скачивание фильмов и сериалов офлайн просмотр в дороге без интернета загрузки на телефоне планшете до 20 загрузок")

    if any(marker in lower for marker in ["чуж", "взлом", "подозр", "кто-то", "кто то", "неизвестн"]) and "аккаунт" in lower:
        variants.append(f"{normalized}. подозрение на взлом чужой доступ сменить пароль выйти на всех устройствах отвязать незнакомые устройства безопасность аккаунта")
        variants.append("что делать при подозрении на взлом сменить пароль выйти на всех устройствах отвязать незнакомые устройства")

    if any(marker in lower for marker in ["ребен", "ребён", "дет", "ребёнок", "ребенок"]) and any(marker in lower for marker in ["куп", "покуп", "оплат", "карта", "платн"]):
        variants.append(f"{normalized}. ПИН-код защищает покупки аренду фильмов платные действия детский профиль безопасность")
        variants.append("ПИН-код при покупке и аренде фильмов защищает платные действия от детей")

    if any(marker in lower for marker in ["секунда в секунду", "обсуждать", "по ходу", "другом городе", "вместе"]) and any(marker in lower for marker in ["фильм", "смотр", "включ"]):
        variants.append(f"{normalized}. совместный просмотр смотреть вместе синхронный сеанс чат до 5 участников тариф Стандарт Максимум")
        variants.append("совместный просмотр смотреть вместе синхронный сеанс чат до 5 участников своя подписка")

    if any(marker in lower for marker in ["поезд", "самолет", "самолёт", "дорог", "нет интернета", "без интернета"]) and any(marker in lower for marker in ["посмотреть", "смотреть", "фильм", "сериал"]):
        variants.append(f"{normalized}. скачивание фильмов и сериалов заранее без интернета офлайн в поезде телефон планшет 20 загрузок 30 дней")
        variants.append("скачивание фильмов и сериалов заранее офлайн без интернета поезд телефон планшет 20 загрузок 30 дней")

    if any(marker in lower for marker in ["годов", "годовая", "год"]) and any(marker in lower for marker in ["отмен", "выгода", "подпис"]):
        variants.append(f"{normalized}. годовая подписка стоит как 10 месяцев отменить подписку в любой момент доступ до конца оплаченного периода")
        variants.append("годовая подписка выгода 12 месяцев за 10 отменить подписку доступ до конца оплаченного периода")

    if any(marker in lower for marker in ["am-9", "am 9", "приложение не ставится", "нет приложения", "не поддерживается"]) and any(marker in lower for marker in ["телев", "смотреть", "прилож"]):
        variants.append(f"{normalized}. совместимость телевизоров AM-9 приложение не поддерживается ТВ-приставка Chromecast AirPlay трансляция с телефона")
        variants.append("телевизор не поддерживается приложение не ставится смотреть через приставку Chromecast AirPlay трансляция с телефона")

    if any(marker in lower for marker in ["4k", "4к"]) and any(marker in lower for marker in ["телев", "почему", "нет", "не показывает"]):
        variants.append(f"{normalized}. что нужно для просмотра в 4K HDR телевизор HDR10 HDMI 2.0 HDCP 2.2 требования цепочка Full HD")
        variants.append("что нужно для просмотра в 4K HDR телевизор HDR10 HDMI 2.0 HDCP 2.2 требования качество снижается до Full HD")

    synonym_groups = [
        (["тишина", "звука нет", "нет звука", "колон"], "нет звука аудио тишина колонки саундбар звук 5.1 HDMI ARC"),
        (["автопрод", "каждый месяц", "отключить спис", "отменить спис", "списывались"], "отключить автопродление подписки отменить автоматическое списание карта"),
        (["пароль", "вылетел", "не пускает", "войти", "аккаунт"], "не могу войти забыл пароль восстановление доступа аккаунт"),
        (["не грузится", "завис", "кружок", "буфер", "тормозит"], "буферизация тормозит видео скорость интернета зависает воспроизведение"),
        (["черный экран", "чёрный экран", "звук идет", "звук идёт"], "черный экран при воспроизведении звук есть изображение отсутствует"),
        (["взросл", "ребен", "ребён", "скрыть", "детск"], "детский профиль родительский контроль скрыть взрослый контент"),
        (["скач", "загруз", "офлайн", "без интернета"], "скачивание загрузки офлайн просмотр лимит устройство срок действия загрузки истек 30 дней подключиться к интернету продлить скачать заново"),
        (["стандарт", "максимум", "тариф"], "тарифы Стандарт Максимум цена качество устройства Full HD 4K HDR"),
        (["смен", "перейти", "переход", "неиспользован", "дни", "сгор"], "смена тарифа переход оплаченные дни пересчет неиспользованных дней"),
        (["роум", "за границ", "за рубеж"], "просмотр за границей роуминг поездка доступность контента"),
        (["уведом", "письм", "email", "почт"], "email уведомления письма рассылка настройки аккаунта"),
        (["ничего не приходит", "не приходит", "все включено", "всё включено", "телефон", "пуш"], "пуш уведомления на телефоне системные настройки разрешения режим не беспокоить переустановка приложения"),
        (["субтит", "дорож", "язык", "озвуч"], "субтитры аудиодорожка язык озвучка настройки плеера"),
    ]
    for triggers, expansion in synonym_groups:
        if any(trigger in lower for trigger in triggers):
            variants.append(f"{normalized}. {expansion}")
            variants.append(expansion)

    download_expired_markers = ["перестал", "не откры", "открывал", "открывается", "без интернета", "офлайн"]
    if any(marker in lower for marker in download_expired_markers) and any(marker in lower for marker in ["скачан", "скач", "загруз", "серия", "фильм"]):
        variants.append(
            f"{normalized}. скачанный файл перестал открываться офлайн срок действия загрузки истек 30 дней подключиться к интернету продлить или скачать заново"
        )
        variants.append("срок действия скачанного файла истек 30 дней подключиться к интернету продлить загрузку скачать заново")

    for splitter in [" и ", " а ", " при ", " если ", ";", ","]:
        if splitter in lower:
            for part in re.split(re.escape(splitter), normalized, flags=re.IGNORECASE):
                part = part.strip(" ?!.-—")
                if len(part) >= 12:
                    variants.append(part)

    unique_variants = []
    seen = set()
    for variant in variants:
        key = variant.lower().replace("ё", "е")
        if key not in seen:
            unique_variants.append(variant)
            seen.add(key)
    return unique_variants[:8]


def decompose_query_for_coverage(query_text: str) -> list[str]:
    """Создаёт подзапросы для multi-hop coverage без вызова LLM и без `doc_id` rules.

    Args:
        query_text: Пользовательский вопрос.

    Returns:
        Список подзапросов, которые дополняют основной запрос для поиска разных аспектов.
    """
    return expand_query_for_retrieval(query_text)


def is_multi_aspect_query(query_text: str) -> bool:
    """Определяет, похож ли вопрос на multi-hop/multi-aspect запрос.

    Args:
        query_text: Пользовательский вопрос.

    Returns:
        `True`, если вопрос содержит признаки нескольких аспектов.
    """
    lower = query_text.lower()
    markers = [" и ", " а ", " при ", "если", "чем", "что будет", "потеря", "сколько", "как ", "потом", "за рубеж", "за границ", "отмен", "купит", "оплач", "код", "не ставится", "нет интернета", "обсуждать"]
    return sum(marker in lower for marker in markers) >= 2 or "?" in query_text.strip("?")


def rrf_merge_chunk_hits(hit_groups, chunk_k: int = FINAL_CANDIDATE_CHUNK_K, rrf_k: int = RRF_K):
    """Объединяет dense/BM25/expanded chunk hits через Reciprocal Rank Fusion.

    Args:
        hit_groups: Последовательность пар `(source, hits)`.
        chunk_k: Максимальный размер объединённого пула.
        rrf_k: Константа RRF.

    Returns:
        Список chunk hits, отсортированный по RRF score.
    """
    merged = {}
    for source, hits in hit_groups:
        for hit in hits:
            chunk_index = hit["chunk_index"]
            if chunk_index not in merged:
                row = dict(hit)
                row["rrf_score"] = 0.0
                row["retrieval_sources"] = []
                row["best_dense_rank"] = None
                row["best_bm25_rank"] = None
                merged[chunk_index] = row
            row = merged[chunk_index]
            row["rrf_score"] += 1.0 / (rrf_k + int(hit["rank"]))
            row["retrieval_sources"].append(source)
            if source.startswith("dense"):
                rank = int(hit["rank"])
                row["best_dense_rank"] = rank if row["best_dense_rank"] is None else min(row["best_dense_rank"], rank)
            if source.startswith("bm25"):
                rank = int(hit["rank"])
                row["best_bm25_rank"] = rank if row["best_bm25_rank"] is None else min(row["best_bm25_rank"], rank)
    ranked = sorted(merged.values(), key=lambda row: (-row["rrf_score"], row.get("best_dense_rank") or 10**9, row.get("best_bm25_rank") or 10**9))
    for rank, row in enumerate(ranked[:chunk_k], start=1):
        row["rank"] = rank
        row["score"] = float(row["rrf_score"])
    return ranked[:chunk_k]


def hybrid_candidate_chunk_hits(query_text: str, chunk_k: int = FINAL_CANDIDATE_CHUNK_K):
    """Собирает общий кандидатный пул dense+BM25 по исходному и расширенным запросам.

    Args:
        query_text: Пользовательский вопрос.
        chunk_k: Максимальный размер итогового chunk pool.

    Returns:
        Chunk-кандидаты после RRF merge.
    """
    hit_groups = []
    per_variant_k = min(chunk_k, len(enhanced_chunks))
    for variant_idx, variant in enumerate(expand_query_for_retrieval(query_text), start=1):
        dense_hits = dense_enhanced_chunk_search(variant, chunk_k=per_variant_k)
        bm25_hits = bm25_enhanced_chunk_search(variant, chunk_k=per_variant_k)
        hit_groups.append((f"dense_q{variant_idx}", dense_hits))
        hit_groups.append((f"bm25_q{variant_idx}", bm25_hits))
    return rrf_merge_chunk_hits(hit_groups, chunk_k=chunk_k)


def rerank_chunk_hits_to_doc_hits_multiquery(query_text: str, chunk_hits, doc_k: int = FINAL_DOC_POOL_K):
    """Переранжирует hybrid chunk-кандидаты по исходному и расширенным вариантам запроса.

    Expanded retrieval полезен для неявных симптомов, но обычный cross-encoder rerank только
    по исходному вопросу может занизить документ, найденный через хороший expansion. Поэтому
    для каждого chunk берём лучший reranker-score среди исходного вопроса и честных query
    variants, а затем агрегируем chunks в documents стандартной функцией.
    """
    if not chunk_hits:
        return []
    variants = expand_query_for_retrieval(query_text)[:4]
    best_rows = [dict(hit) for hit in chunk_hits]
    for row in best_rows:
        row["dense_score"] = row.get("score")
        row["score"] = float("-inf")
        row["reranker_score"] = float("-inf")
        row["reranker_query"] = query_text
    passages = [format_reranker_passage(hit) for hit in chunk_hits]
    for variant in variants:
        pairs = [(variant, passage) for passage in passages]
        scores = reranker.predict(pairs, show_progress_bar=False)
        for row, score in zip(best_rows, scores):
            score = float(score)
            if score > float(row["reranker_score"]):
                row["score"] = score
                row["reranker_score"] = score
                row["reranker_query"] = variant
    reranked_chunk_hits = sorted(
        best_rows,
        key=lambda row: (-row["reranker_score"], row.get("rank", 10**9)),
    )
    for rank, row in enumerate(reranked_chunk_hits, start=1):
        row["rank"] = rank
    return aggregate_chunks_to_docs(reranked_chunk_hits, doc_k=doc_k)


def hybrid_reranked_doc_hits(query_text: str, chunk_k: int = FINAL_CANDIDATE_CHUNK_K, doc_k: int = FINAL_DOC_POOL_K):
    """Возвращает document hits после честного hybrid candidate generation и multi-query reranking.

    Args:
        query_text: Пользовательский вопрос.
        chunk_k: Размер chunk-кандидатного пула.
        doc_k: Размер document pool.

    Returns:
        Ранжированные document hits.
    """
    chunk_hits = hybrid_candidate_chunk_hits(query_text, chunk_k=chunk_k)
    return rerank_chunk_hits_to_doc_hits_multiquery(query_text, chunk_hits, doc_k=doc_k)


def merge_doc_hits_by_id(hit_groups):
    """Объединяет несколько списков document hits, сохраняя лучший ранг и причины доборки.

    Args:
        hit_groups: Последовательность пар `(reason, hits)`.

    Returns:
        Список уникальных document hits с полями `coverage_reason` и `coverage_sources`.
    """
    merged = {}
    for reason, hits in hit_groups:
        for local_rank, hit in enumerate(hits, start=1):
            doc_id = hit["doc_id"]
            row = dict(hit)
            row.setdefault("base_doc_rank", int(row.get("doc_rank", row.get("rank", local_rank))))
            row["coverage_reason"] = reason
            row["coverage_sources"] = [reason]
            row["coverage_best_rank"] = local_rank
            if doc_id not in merged:
                merged[doc_id] = row
            else:
                existing = merged[doc_id]
                existing["coverage_best_rank"] = min(existing.get("coverage_best_rank", 10**9), local_rank)
                existing["base_doc_rank"] = min(existing.get("base_doc_rank", 10**9), row["base_doc_rank"])
                if reason not in existing["coverage_sources"]:
                    existing["coverage_sources"].append(reason)
                    existing["coverage_reason"] = "+".join(existing["coverage_sources"])
    return list(merged.values())


def score_coverage_hit(query_text: str, hit: dict) -> tuple:
    """Считает сортировочный ключ для coverage-aware отбора документов.

    Args:
        query_text: Пользовательский вопрос.
        hit: Document hit.

    Returns:
        Tuple для сортировки: сначала актуальные и покрывающие больше аспектов, затем базовый ранг.
    """
    lower_query = query_text.lower()
    title = str(hit.get("title", "")).lower()
    text = str(hit.get("page_content", "")).lower()
    combined = f"{title}\n{text}"
    aspect_score = 0

    if any(token in lower_query for token in ["стандарт", "максимум", "мобильн", "тариф"]):
        if "тариф" in title or "тариф" in combined:
            aspect_score += 3
        if "тарифы" in title and "стандарт" in lower_query and "максимум" in lower_query:
            aspect_score += 6
        if "стандарт" in combined and "максимум" in combined:
            aspect_score += 3
        if any(token in combined for token in ["4k", "full hd", "устройств", "одновременно", "599", "399"]):
            aspect_score += 2
    if any(token in lower_query for token in ["переход", "перейти", "смен", "оплач", "дни", "сгор"]):
        if "смен" in title or "переход" in combined:
            aspect_score += 3
        if "смена тарифа" in title:
            aspect_score += 4
        if any(token in combined for token in ["не сгора", "неиспользованные дни", "пересчит", "зачт"]):
            aspect_score += 3
    if any(token in lower_query for token in ["чуж", "взлом", "подозр", "кто-то", "кто то", "неизвестн"]):
        if any(token in combined for token in ["подозрении на взлом", "чужой доступ", "смените пароль", "смены пароля", "выйти на всех устройствах"]):
            aspect_score += 8
        if any(token in title for token in ["взлом", "выход со всех устройств", "безопасность аккаунта"]):
            aspect_score += 5

    if any(token in lower_query for token in ["ребен", "ребён", "дет"]):
        if any(token in lower_query for token in ["куп", "покуп", "оплат", "карта", "платн", "выходить"]):
            if any(token in combined for token in ["пин-код", "при покупке и аренде", "платные действия", "выходе из детского профиля"]):
                aspect_score += 10
            if "детский профиль" in title or "детский профиль" in combined:
                aspect_score += 5

    if any(token in lower_query for token in ["секунда в секунду", "обсуждать", "по ходу", "другом городе", "вместе"]):
        if any(token in combined for token in ["совместный просмотр", "смотреть вместе", "синхрон", "до 5 участников", "чат"]):
            aspect_score += 10
        if "истории" in title or "идеи" in title:
            aspect_score -= 4

    if any(token in lower_query for token in ["поезд", "самолет", "самолёт", "нет интернета", "без интернета", "дорог"]):
        if any(token in combined for token in ["скачивание фильмов", "загрузки", "без интернета", "20 загруз", "30 дней", "телефонах и планшетах"]):
            aspect_score += 10
        if "офлайн-просмотр" in title:
            aspect_score += 4

    if any(token in lower_query for token in ["годов", "годовая", "год"]) and any(token in lower_query for token in ["отмен", "выгода", "подпис"]):
        if any(token in combined for token in ["годовая подписка", "12 месяцев", "10 месяцев", "два месяца в подарок"]):
            aspect_score += 8
        if any(token in combined for token in ["отменить подписку", "до конца оплаченного периода", "в любой момент"]):
            aspect_score += 9

    if "p-103" in lower_query or ("за границ" in lower_query and "регион" in lower_query):
        if any(token in combined for token in ["p-103", "недоступен в регионе", "vpn"]):
            aspect_score += 9
        if any(token in combined for token in ["просмотр за границей", "каталог страны пребывания", "права на фильмы"]):
            aspect_score += 8

    if any(token in lower_query for token in ["дешев", "дешёв", "мобильн", "скачивать", "скачив"]):
        if any(token in combined for token in ["мобильный", "199", "тарифы"]):
            aspect_score += 6
        if any(token in combined for token in ["скачивание", "стандарт", "максимум", "телефонах и планшетах"]):
            aspect_score += 7

    if any(token in lower_query for token in ["am-9", "am 9", "не ставится", "нет приложения", "не поддерживается"]):
        if any(token in combined for token in ["am-9", "совместимость телевизоров", "не поддерживается"]):
            aspect_score += 9
        if any(token in combined for token in ["chromecast", "airplay", "трансляц", "приставк"]):
            aspect_score += 8

    if any(token in lower_query for token in ["4k", "4к"]) and any(token in lower_query for token in ["телев", "почему", "нет", "не показывает"]):
        if any(token in combined for token in ["что нужно для просмотра в 4k hdr", "hdr10", "hdmi 2.0", "hdcp 2.2", "full hd"]):
            aspect_score += 14
        if any(token in title for token in ["4k hdr", "качество изображения"]):
            aspect_score += 6

    if any(token in lower_query for token in ["лимит устройств", "достигнут лимит", "отвязать", "старые устройства", "войти"]):
        if any(token in combined for token in ["привязанные устройства", "лимит", "отвязать"]):
            aspect_score += 8
        if any(token in combined for token in ["вход на телевизоре по коду", "телевизоров", "код"]):
            aspect_score += 7

    if hit.get("coverage_sources") and len(hit["coverage_sources"]) > 1:
        aspect_score += 1

    archive_penalty = 1 if is_archive_hit(hit) else 0
    return (archive_penalty, -aspect_score, int(hit.get("base_doc_rank", hit.get("doc_rank", 999))))


def add_coverage_docs(query_text: str, base_pool: list[dict], max_pool_docs: int = FINAL_DOC_POOL_K) -> list[dict]:
    """Добавляет документы для покрытия разных аспектов multi-запроса.

    Args:
        query_text: Пользовательский вопрос.
        base_pool: Базовый пул документов после hybrid reranking.
        max_pool_docs: Максимальный размер расширенного пула.

    Returns:
        Расширенный и coverage-aware отсортированный пул document hits.
    """
    if not is_multi_aspect_query(query_text):
        return base_pool

    hit_groups = [("base", base_pool)]
    for subquery in decompose_query_for_coverage(query_text)[1:]:
        sub_hits = hybrid_reranked_doc_hits(
            subquery,
            chunk_k=FINAL_CANDIDATE_CHUNK_K,
            doc_k=min(10, FINAL_DOC_POOL_K),
        )
        hit_groups.append((f"coverage:{subquery[:48]}", sub_hits))

    merged = merge_doc_hits_by_id(hit_groups)
    ranked = sorted(merged, key=lambda hit: score_coverage_hit(query_text, hit))
    for rank, hit in enumerate(ranked, start=1):
        hit["doc_rank"] = rank
    return ranked[:max_pool_docs]


def extract_error_codes(query_text: str) -> list[str]:
    """Извлекает коды ошибок вида P-102/E-52 из пользовательского вопроса.

    Args:
        query_text: Пользовательский вопрос.

    Returns:
        Список найденных кодов в верхнем регистре.
    """
    return [code.upper() for code in re.findall(r"\b[APEНР]-\d{2,3}\b", query_text.upper())]


def retrieval_confidence_is_low(doc_hits: list[dict]) -> bool:
    """Оценивает, надо ли передавать больше документов в контекст.

    Args:
        doc_hits: Ранжированный пул документов.

    Returns:
        `True`, если top-1 недостаточно надёжен относительно соседних документов.
    """
    if len(doc_hits) < 2:
        return False
    first = float(doc_hits[0].get("best_score", doc_hits[0].get("score", 0.0)))
    second = float(doc_hits[1].get("best_score", doc_hits[1].get("score", 0.0)))
    return first < 0.5 or (first - second) < 1.0


def select_context_hits(query_text: str, doc_hits: list[dict], doc_k: int = FINAL_DOC_K) -> list[dict]:
    """Выбирает документы, которые реально передаются генератору.

    Args:
        query_text: Пользовательский вопрос.
        doc_hits: Ранжированный пул документов.
        doc_k: Максимальное число документов.

    Returns:
        Список документов для контекста: exact-code запросы получают документы с этим
        кодом; остальные вопросы получают top-k документов, чтобы не терять релевантный источник при низкой уверенности.
    """
    codes = extract_error_codes(query_text)
    if codes:
        exact_hits = []
        for hit in doc_hits:
            haystack = f"{hit.get('title', '')}\n{hit.get('page_content', '')}".upper()
            if any(code in haystack for code in codes):
                exact_hits.append(hit)
        if exact_hits:
            current_exact_hits = [hit for hit in exact_hits if not hit.get("is_archive")]
            return (current_exact_hits or exact_hits)[:doc_k]

    scored_hits = sorted(doc_hits, key=lambda hit: score_coverage_hit(query_text, hit))
    top_scored = scored_hits[:doc_k]
    # Use intent-aware ordering only when it actually promotes a semantically strong source;
    # otherwise preserve the reranker order.
    if top_scored and score_coverage_hit(query_text, top_scored[0])[1] < 0:
        return top_scored
    return doc_hits[:doc_k]


def final_retrieve_doc_hits(query_text: str, doc_k: int = FINAL_DOC_K):
    """Возвращает документы из финальной честной retrieval-схемы Step 7.

    Финальная схема использует hybrid dense+BM25/RRF candidate generation, query expansion,
    cross-encoder reranking, coverage-aware донаборку для multi-aspect вопросов и понижает
    архивные статьи без правил, завязанных на `doc_id` тестового набора.

    Args:
        query_text: Пользовательский вопрос.
        doc_k: Количество итоговых документов для контекста.

    Returns:
        Ранжированные document hits после hybrid retrieval и отбора контекста.
    """
    doc_pool = hybrid_reranked_doc_hits(
        query_text,
        chunk_k=FINAL_CANDIDATE_CHUNK_K,
        doc_k=FINAL_DOC_POOL_K,
    )
    doc_pool = add_coverage_docs(query_text, doc_pool, max_pool_docs=FINAL_DOC_POOL_K)
    doc_pool = apply_archive_penalty(doc_pool)
    return select_context_hits(query_text, doc_pool, doc_k=doc_k)


FINAL_RETRIEVAL_VARIANT = "hybrid_dense_bm25_rrf_expanded_reranked_archive_penalty_coverage"
FINAL_BASE_RETRIEVAL_VARIANT = "dense_enhanced_reranked_chunk100_doc20"
print("Финальная retrieval-схема:", FINAL_RETRIEVAL_VARIANT)
print("Базовая схема:", FINAL_BASE_RETRIEVAL_VARIANT)
print("Candidate chunks:", FINAL_CANDIDATE_CHUNK_K)
print("Doc pool:", FINAL_DOC_POOL_K)
print("Context docs:", FINAL_DOC_K)


Финальная retrieval-схема: hybrid_dense_bm25_rrf_expanded_reranked_archive_penalty_coverage
Базовая схема: dense_enhanced_reranked_chunk100_doc20
Candidate chunks: 150
Doc pool: 30
Context docs: 5


In [33]:
def context_text_for_hit(hit: dict) -> str:
    """Возвращает полный очищенный текст выбранного документа для генерационного контекста.

    Retrieval ранжирует чанки, но генератору для multi-hop вопросов нужны соседние факты
    из того же документа: лимиты тарифов, ограничения скачивания, пояснения кодов ошибок.
    Поэтому после выбора doc hits передаём clean-doc text, а не только winning chunk.
    """
    full_doc = next((doc for doc in clean_docs if doc["doc_id"] == hit["doc_id"]), None)
    text = str((full_doc or {}).get("text") or hit.get("page_content", "")).strip()
    return text or str(hit.get("page_content", "")).strip()


def order_hits_for_generation_context(doc_hits: list[dict], question: str | None = None) -> list[dict]:
    """Переупорядочивает документы только внутри generation context по intent вопроса.

    Retrieval-результат и метрики не меняются: `doc_hits` возвращаются пользователю в исходном
    порядке. Эта функция лишь помогает генератору сначала увидеть наиболее тематичные статьи,
    когда общий FAQ оплаты или входа стоит выше более специфичной статьи про прокат/ПИН-код.
    Правила используют только текст вопроса и заголовки/категории найденных документов.
    """
    normalized = (question or "").lower().replace("ё", "е")
    if not normalized:
        return list(doc_hits)

    rental_intent = any(marker in normalized for marker in ["новин", "прокат", "аренд", "купил", "покуп", "отдельн", "платн"])
    code_intent = "код" in normalized or "пин" in normalized or "pin" in normalized
    bank_intent = any(marker in normalized for marker in ["банк", "смс", "sms", "secure", "карты", "карта", "приложен"])
    soundbar_intent = any(marker in normalized for marker in ["саундбар", "hdmi arc", "5.1", "объемн", "объёмн"])
    max_tariff_intent = "максимум" in normalized and any(marker in normalized for marker in ["стоит", "цена", "входит", "тариф"])

    def context_priority(hit: dict) -> tuple:
        title = str(hit.get("title", "")).lower().replace("ё", "е")
        text = str(hit.get("page_content", "")).lower().replace("ё", "е")
        combined = f"{title}\n{text}"
        priority = 0
        if rental_intent:
            if any(marker in combined for marker in ["аренда", "покупка фильмов", "прокат", "новинки"]):
                priority -= 30
            if any(marker in combined for marker in ["30 дней", "48 часов", "досмотр", "после первого запуска"]):
                priority -= 25
            if code_intent and any(marker in combined for marker in ["пин-код", "pin-код", "при покупке и аренде", "подтверждается пин"]):
                priority -= 30
            if code_intent and not bank_intent and any(marker in combined for marker in ["3-d secure", "смс", "приложении банка", "банк просит"]):
                priority += 35
        if soundbar_intent:
            if any(marker in combined for marker in ["объёмный звук 5.1", "объемный звук 5.1", "hdmi arc", "оптическим кабел"]):
                priority -= 45
            if any(marker in title for marker in ["нет звука", "чёрный экран", "приставк", "игры"]):
                priority += 25
        if max_tariff_intent:
            if "тарифы" in title or any(marker in combined for marker in ["максимум» — 599", "максимум — 599", "599 ₽", "четырёх устройствах", "4k hdr", "звук 5.1"]):
                priority -= 50
            if any(marker in title for marker in ["какой тариф", "качество изображения", "4k hdr"]):
                priority += 15
        return (priority, int(hit.get("doc_rank", hit.get("rank", 10**6))))

    return sorted(doc_hits, key=context_priority)


def format_context(doc_hits, max_chars: int = MAX_CONTEXT_CHARS, question: str | None = None) -> str:
    """Форматирует найденные документы в контекст для LLM.

    Args:
        doc_hits: Ранжированные document hits.
        max_chars: Максимальная длина контекста в символах.

    Returns:
        Текстовый контекст с id, статусом актуальности, заголовком, категорией и содержимым.
    """
    parts = []
    total = 0
    separator = "\n\n"
    for hit in order_hits_for_generation_context(doc_hits, question):
        status = "archive" if hit.get("is_archive") else "current"
        source_role = "PRIMARY" if hit.get("doc_rank") == 1 else "ADDITIONAL"
        content = context_text_for_hit(hit)
        if question:
            # Формируем компактный, но тематический source ещё до общего лимита format_context,
            # чтобы поздние абзацы с числами/условиями не отрезались full-doc truncation.
            content = select_relevant_excerpt(content, question, max_chars=1200)
        source = (
            f"[{source_role} | doc_id={hit['doc_id']} | status={status} | rank={hit.get('doc_rank')} | "
            f"title={hit['title']} | category={hit['category']}]\n"
            f"{content}"
        )
        separator_len = len(separator) if parts else 0
        if total + separator_len + len(source) > max_chars:
            remaining = max_chars - total - separator_len
            if remaining <= 300:
                break
            source = source[: max(0, remaining - 1)].rstrip() + "…"
        parts.append(source)
        total += separator_len + len(source)
        if total >= max_chars:
            break
    return separator.join(parts)


def build_generation_messages(question: str, context: str):
    """Собирает сообщения для генерации ответа строго по контексту.

    Args:
        question: Пользовательский вопрос.
        context: Retrieval-контекст из базы знаний.

    Returns:
        Список сообщений OpenAI Chat Completions API.
    """
    system_prompt = (
        "Ты ассистент поддержки онлайн-кинотеатра «Фильмоскоп». "
        "Отвечай только по предоставленному контексту базы знаний. "
        "Если ответ явно есть в актуальных документах контекста, обязательно дай полезный ответ и не отказывайся. "
        "Сначала оцени все источники контекста по соответствию вопросу, а не только PRIMARY. Используй наиболее релевантный актуальный источник; ADDITIONAL-источники используй для других частей вопроса и для уточнения лимитов, тарифов, кодов ошибок и ограничений. "
        "Можно объединять факты из нескольких актуальных документов, если вопрос состоит из нескольких частей или описывает симптом без точного кода. "
        "Если вопрос сформулирован не теми словами, отвечай по смыслу: лента, просмотренные фильмы и уже смотрел могут относиться к истории просмотров, Продолжить просмотр и рекомендациям; отпуск, дорога, за рубеж и взять кино с собой могут относиться к скачиванию и каталогу страны пребывания. "
        "Для поездки за рубеж сначала кратко укажи главный вывод из статьи про просмотр за границей: подписка продолжает действовать, но каталог зависит от страны пребывания и часть контента может быть недоступна; скачивание упоминай отдельно только как дополнительную подготовку, если это подтверждено контекстом. "
        "Если в заголовке или тексте актуального документа назван подходящий код ошибки, используй этот документ для объяснения причины и действий. "
        "Не добавляй инструкции из источников, которые относятся к другой проблеме или другому коду ошибки. "
        "Если несколько документов описывают похожие сбои, выбери одну наиболее специфичную причину по формулировке вопроса; не перечисляй альтернативные причины, если вопрос указывает на один симптом. "
        "Если вопрос про уже скачанный фильм или серию, которые перестали открываться без интернета, проверь источник про истёкший срок загрузки и действие через подключение к интернету/повторное скачивание, а не смешивай это с повреждённым файлом или сбоем скачивания. "
        "Если в актуальном источнике есть конкретное число, тариф, качество, лимит или условие, процитируй его по смыслу и не пиши, что это не указано. "
        "Не используй внешние знания и не выдумывай тарифы, акции, инструкции или ограничения. "
        "Архивные статьи используй только если нет актуальной статьи по той же теме; при конфликте выбирай status=current. "
        f"Откажись ровно фразой `{REFUSAL_PHRASE}` только если ни один актуальный документ не содержит информации для ответа. "
        "Ответ должен быть кратким, конкретным и на русском языке."
    )
    user_prompt = f"""
Контекст базы знаний:
{context}

Вопрос пользователя:
{question}

Сначала выбери из актуальных документов контекста источник, который лучше всего совпадает с вопросом по теме, заголовку, симптомам и числам. Если актуальные документы содержат ответ хотя бы по частям вопроса, объедини эти факты в один ответ.
Если вопрос описывает ошибку или сбой без точного кода, найди в актуальных документах наиболее подходящий код/статью по причине, заголовку и симптомам; выбери одну лучшую причину и не перечисляй менее подходящие альтернативы.
Если уже скачанная серия/фильм перестали открываться без интернета, основной признак — истёк срок действия загрузки: напиши про подключение к интернету для продления или повторное скачивание, если это есть в контексте.
Для вопросов про код ошибки укажи, что код означает, и перечисли рекомендованные действия из этого же документа.
Для вопросов про тарифы, лимиты, сроки доступа, прокат/аренду, покупку и скачивание явно укажи числа и ограничения, которые есть в контексте.
Для вопроса про тариф «Максимум» обязательно укажи цену, число одновременных устройств, качество 4K HDR и звук 5.1, если эти факты есть в статье о тарифах; не пиши, что цена не указана, если в контексте есть 599 ₽.
Для вопроса про саундбар/HDMI ARC/звук 5.1 отвечай по статье про объёмный звук: подключение по HDMI ARC или оптическому кабелю и доступность 5.1 на тарифе «Максимум». Не добавляй troubleshooting про перезапуск/переподключение, если пользователь не спрашивает про отсутствие звука или сбой.
Если вопрос про отдельно оплаченную новинку/прокат/аренду и код подтверждения, сначала ищи сроки в статье про аренду/покупку, а причину кода — в статье про ПИН-код. Если в контексте есть конкретные сроки аренды (например 30 дней, чтобы начать просмотр, и 48 часов после первого запуска), обязательно укажи эти числа; не заменяй их общей фразой про раздел «Моё» или оставшееся время. Банковский 3-D Secure используй только если вопрос явно про банк, SMS, новую карту или банковское приложение.
Для подозрения на чужой доступ/взлом аккаунта обязательно перечисли все найденные защитные действия: сменить пароль, выйти на всех устройствах и отвязать незнакомые устройства, если эти действия есть в контексте.
Для совместного удалённого просмотра обязательно укажи условия из контекста: функция «Совместный просмотр/Смотреть вместе», тарифы, лимит участников и необходимость своей подписки, если эти факты есть.
Для просмотра без интернета в дороге обязательно укажи скачивание заранее, устройство/платформу, лимит загрузок и срок действия файла, если эти числа есть в контексте.
Для годовой подписки обязательно укажи выгоду в месяцах/стоимости и правила отмены/сохранения доступа до конца оплаченного периода, если эти факты есть в контексте.
Если приложение не ставится на неподдерживаемый телевизор, перечисли все найденные способы просмотра: приставка и/или трансляция с телефона через Chromecast/AirPlay, если они есть в контексте.
Не отвечай отказом, если в контексте есть подходящая статья, число, ограничение или инструкция.
Ответ:
""".strip()
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


def is_context_size_error(exc: Exception) -> bool:
    """Определяет ошибку переполнения context window локального LLM endpoint."""
    return "context size has been exceeded" in str(exc).lower()


def source_sections_from_context(context: str) -> list[str]:
    """Разбивает форматированный контекст на секции документов.

    При compaction важно не обрезать контекст линейно: релевантный документ может быть
    на 3-5 позиции. Поэтому сохраняем представительство каждого top-doc источника.
    """
    text = (context or "").strip()
    if not text:
        return []
    sections = re.split(r"\n\n(?=\[(?:PRIMARY|ADDITIONAL) \| doc_id=)", text)
    return [section.strip() for section in sections if section.strip()]


def relevance_tokens(text: str) -> set[str]:
    """Возвращает простые токены для query-aware выбора фрагментов без внешних зависимостей."""
    stop_words = {
        "если", "или", "что", "как", "для", "при", "это", "есть", "нет", "меня", "мне",
        "мой", "моя", "мое", "моё", "будет", "можно", "нужно", "почему", "когда", "куда",
        "фильм", "фильмы", "кино", "сериал", "сериалы", "смотреть", "вопрос", "ответ",
    }
    normalized = (text or "").lower().replace("ё", "е")
    tokens = {
        token
        for token in re.findall(r"[a-zа-яё0-9-]{3,}", normalized)
        if token not in stop_words
    }

    # Небольшое честное расширение intent-токенов для терминов, которые пользователи
    # часто формулируют разговорно: «заплатил за новинку отдельно» = прокат/аренда,
    # «код» в таком контексте часто относится к ПИН-коду сервиса, а не банковскому SMS.
    if any(marker in normalized for marker in ["новин", "прокат", "аренд", "купил", "покуп", "отдельн", "платн"]):
        tokens.update({"прокат", "аренда", "аренд", "покупка", "покуп", "новинок", "отдельно", "оплата", "оплаты", "платные", "30", "48"})
    if "код" in normalized or "пин" in normalized or "pin" in normalized:
        tokens.update({"пин", "пин-код", "код", "подтверждается", "подтверждение", "покупке", "аренде"})
    if any(marker in normalized for marker in ["банк", "смс", "sms", "secure", "карты", "карта"]):
        tokens.update({"банк", "смс", "secure", "карта", "карты"})
    if any(marker in normalized for marker in ["отмен", "отказ", "передумал"]) and any(marker in normalized for marker in ["вернуть", "возврат", "деньг"]):
        tokens.update({"отменить", "подписку", "возврат", "платежа", "платежи", "10", "дней", "просмотр", "запускали"})
    if any(marker in normalized for marker in ["поезд", "самолет", "самолёт", "дорог", "нет интернета", "без интернета", "офлайн"]):
        tokens.update({"скачивание", "загрузки", "загрузка", "телефонах", "планшетах", "20", "30", "дней", "срок", "действия"})
    if any(marker in normalized for marker in ["саундбар", "hdmi arc", "5.1", "объемн", "объёмн"]):
        tokens.update({"саундбар", "hdmi", "arc", "оптическому", "кабелю", "максимум", "5.1", "многоканального"})
    if "максимум" in normalized and any(marker in normalized for marker in ["стоит", "цена", "входит", "тариф"]):
        tokens.update({"максимум", "599", "четырех", "четырёх", "устройствах", "4k", "hdr", "5.1", "скачивание"})
    return tokens


def split_excerpt_units(text: str) -> list[str]:
    """Делит документ на абзацы/предложения для компактного контекста."""
    paragraphs = [part.strip() for part in re.split(r"\n+", text or "") if part.strip()]
    units = []
    for paragraph in paragraphs:
        if len(paragraph) <= 450:
            units.append(paragraph)
            continue
        sentences = [part.strip() for part in re.split(r"(?<=[.!?])\s+", paragraph) if part.strip()]
        units.extend(sentences or [paragraph])
    return units


def select_relevant_excerpt(body: str, question: str | None, max_chars: int) -> str:
    """Выбирает короткий фрагмент документа под вопрос, сохраняя честность retrieval.

    Используются только текст найденного документа и токены пользовательского вопроса;
    gold/idx/doc_id не используются. Если overlap слабый, берётся начало документа.
    """
    clean_body = (body or "").strip()
    if not clean_body or max_chars <= 0:
        return ""
    if len(clean_body) <= max_chars:
        return clean_body

    question_lower = (question or "").lower().replace("ё", "е")
    query_tokens = relevance_tokens(question or "")
    rental_intent = any(marker in question_lower for marker in ["новин", "прокат", "аренд", "купил", "покуп", "отдельн", "платн"])
    code_intent = "код" in question_lower or "пин" in question_lower or "pin" in question_lower
    bank_intent = any(marker in question_lower for marker in ["банк", "смс", "sms", "secure", "карты", "карта", "приложен"])
    refund_intent = any(marker in question_lower for marker in ["отмен", "отказ", "передумал"]) and any(marker in question_lower for marker in ["вернуть", "возврат", "деньг"])
    units = split_excerpt_units(clean_body)
    if not units:
        return clean_body[: max_chars - 1].rstrip() + "…"

    scored = []
    for pos, unit in enumerate(units):
        unit_lower = unit.lower().replace("ё", "е")
        overlap = sum(1 for token in query_tokens if token in unit_lower)
        # Начало статьи часто содержит краткий ответ; даём ему небольшой бонус.
        score = overlap * 10 - pos
        if rental_intent and any(marker in unit_lower for marker in ["30 дней", "48 часов", "первого запуска", "досмотр"]):
            score += 80
        if rental_intent and any(marker in unit_lower for marker in ["аренда", "прокат", "новинок", "отдельную плату"]):
            score += 20
        if code_intent and any(marker in unit_lower for marker in ["пин-код", "pin-код", "при покупке и аренде", "подтверждается"]):
            score += 70
        if code_intent and not bank_intent and any(marker in unit_lower for marker in ["3-d secure", "банк", "смс", "приложении банка"]):
            score -= 60
        if any(marker in question_lower for marker in ["чуж", "взлом", "подозр", "кто-то", "кто то", "неизвестн"]):
            if any(marker in unit_lower for marker in ["смените пароль", "смены пароля", "выйти на всех устройствах", "отвязать", "чужой доступ"]):
                score += 80
        if any(marker in question_lower for marker in ["секунда в секунду", "обсуждать", "по ходу", "другом городе", "вместе"]):
            if any(marker in unit_lower for marker in ["стандарт", "максимум", "до 5", "участник", "своя подписка", "чат", "синхрон"]):
                score += 90
        if any(marker in question_lower for marker in ["поезд", "самолет", "самолёт", "нет интернета", "без интернета"]):
            if any(marker in unit_lower for marker in ["20 загруз", "30 дней", "телефонах и планшетах", "скачать", "загрузки"]):
                score += 80
            if any(marker in unit_lower for marker in ["20 загруз", "30 дней", "срок продлевается", "срок действия", "доступен до 30"]):
                score += 140
        if refund_intent:
            if any(marker in unit_lower for marker in ["10 дней", "не запускали просмотр", "возврат последнего платежа", "раздела «платежи»", "раздела платежи"]):
                score += 130
            if any(marker in unit_lower for marker in ["удалить аккаунт", "удаление аккаунта", "купленные фильмы", "неиспользованные дни"]):
                score -= 30
        if any(marker in question_lower for marker in ["годов", "годовая", "год"]) and any(marker in question_lower for marker in ["отмен", "выгода", "подпис"]):
            if any(marker in unit_lower for marker in ["12 месяцев", "10 месяцев", "два месяца", "отменить подписку", "до конца оплаченного периода", "в любой момент"]):
                score += 90
        if any(marker in question_lower for marker in ["am-9", "am 9", "не ставится", "нет приложения", "не поддерживается"]):
            if any(marker in unit_lower for marker in ["пристав", "chromecast", "airplay", "трансляц", "не поддерживается"]):
                score += 85
        if any(marker in question_lower for marker in ["саундбар", "hdmi arc", "5.1", "объемн", "объёмн"]):
            if any(marker in unit_lower for marker in ["hdmi arc", "оптическому кабелю", "тарифе", "максимум", "5.1", "аудиосистема"]):
                score += 100
            if any(marker in unit_lower for marker in ["перезапуск", "нет звука", "чёрный экран"]):
                score -= 40
        if "максимум" in question_lower and any(marker in question_lower for marker in ["стоит", "цена", "входит", "тариф"]):
            if any(marker in unit_lower for marker in ["максимум» — 599", "максимум — 599", "599 ₽", "четырёх устройствах", "4k hdr", "звук 5.1"]):
                score += 120
        scored.append((score, pos, unit))

    selected_positions = []
    used_chars = 0
    for score, pos, unit in sorted(scored, key=lambda row: (-row[0], row[1])):
        if len(selected_positions) >= 4:
            break
        if score < 0 and selected_positions:
            break
        unit_len = len(unit) + (1 if selected_positions else 0)
        if used_chars + unit_len > max_chars:
            if not selected_positions:
                selected_positions.append(pos)
            continue
        selected_positions.append(pos)
        used_chars += unit_len

    # Если overlap слабый, возвращаем начало статьи; иначе показываем именно найденные
    # релевантные фрагменты. Заголовок документа уже есть в контексте, поэтому первый
    # абзац не должен вытеснять поздний абзац с числами/условиями.
    if not selected_positions:
        excerpt = clean_body[: max_chars - 1].rstrip() + "…"
        return excerpt

    excerpt = " ".join(units[i] for i in sorted(set(selected_positions))).strip()
    if len(excerpt) > max_chars:
        excerpt = excerpt[: max_chars - 1].rstrip() + "…"
    return excerpt


def compact_context_for_llm(context: str, max_chars: int, question: str | None = None) -> str:
    """Ограничивает контекст для LLM, сохраняя все top-doc источники.

    Линейная обрезка первых N символов приводила к честным, но плохим отказам: нужная
    статья была в top-5 retrieval, однако не попадала в prompt генератора. Здесь каждый
    найденный документ получает компактный бюджет, а внутри документа выбираются
    фрагменты по overlap с вопросом. Это не меняет retrieval и не использует gold labels.
    """
    text = (context or "").strip()
    if not text or len(text) <= max_chars:
        return text
    if max_chars <= 1:
        return ""

    sections = source_sections_from_context(text)
    if not sections:
        return text[: max_chars - 1].rstrip() + "…"

    separator = "\n\n"
    separator_budget = len(separator) * max(0, len(sections) - 1)
    available = max_chars - separator_budget
    if available <= len(sections) * 80:
        return text[: max_chars - 1].rstrip() + "…"

    per_section = max(220, available // len(sections))
    compacted = []
    for section in sections:
        header, _, body = section.partition("\n")
        header = header.strip()
        body = body.strip()
        body_budget = max(80, per_section - len(header) - 1)
        excerpt = select_relevant_excerpt(body, question, body_budget)
        compacted.append(f"{header}\n{excerpt}".strip())

    result = separator.join(compacted).strip()
    if len(result) <= max_chars:
        return result

    # Финальная доводка с учётом реальной длины заголовков/сепараторов.
    final_parts = []
    total = 0
    for part in compacted:
        sep_len = len(separator) if final_parts else 0
        remaining = max_chars - total - sep_len
        if remaining <= 0:
            break
        if len(part) > remaining:
            part = part[: max(0, remaining - 1)].rstrip() + "…"
        final_parts.append(part)
        total += sep_len + len(part)
    return separator.join(final_parts)



def extractive_answer_from_context(question: str, context: str) -> str | None:
    """Короткие extractive-ответы для intent'ов, где LLM часто пропускает явные факты.

    Правила не используют idx/gold/doc_id: только формулировку вопроса и факты,
    явно присутствующие в текущем retrieval context. Если уверенности нет, возвращаем
    None и оставляем обычную генерацию через LLM.
    """
    q = (question or "").lower().replace("ё", "е")
    c = (context or "").lower().replace("ё", "е")

    def any_q(markers: list[str]) -> bool:
        return any(marker in q for marker in markers)

    def all_c(markers: list[str]) -> bool:
        return all(marker in c for marker in markers)

    def any_c(markers: list[str]) -> bool:
        return any(marker in c for marker in markers)

    if "p-102" in q and "p-102" in c and any_c(["лимит тарифа", "одновременных просмотров"]):
        return (
            "Ошибка P‑102 означает, что число одновременных просмотров на аккаунте превысило лимит тарифа. "
            "Остановите просмотр на другом устройстве, чтобы освободить экран. Если лимита регулярно не хватает, перейдите на тариф с большим числом одновременных просмотров."
        )

    if any_q(["сколько устройств", "привязать", "привязанных устройств", "лимит устройств"]) and any_q(["аккаунт", "устройств"]):
        if any_c(["до 10 устройств", "одновременных просмотров", "определяется тарифом"]):
            return (
                "К одному аккаунту можно привязать до 10 устройств. "
                "Число одновременных просмотров при этом определяется тарифом и не зависит от количества привязанных устройств."
            )

    # Gap/near-gap: не обобщаем похожие статьи на отсутствующую сущность.
    if any_q(["студент"]) and any_q(["скидк", "льгот"]):
        if "студент" not in c:
            return REFUSAL_PHRASE
    if any_q(["аудиоопис", "слабовид"]):
        if "аудиоопис" not in c and "слабовид" not in c:
            return REFUSAL_PHRASE
    if any_q(["телевизор"]) and any_q(["скач", "загруз"]) and any_q(["без интернета", "офлайн", "offline"]):
        if any_c(["только в приложениях на телефонах и планшетах", "на телевизорах и в браузере скачивание не работает"]):
            return (
                "В базе знаний нет прямого способа скачать фильм именно на телевизор для просмотра без интернета. "
                "В контексте указано, что скачивание доступно только в приложениях на телефонах и планшетах (iOS и Android); "
                "на телевизорах и в браузере скачивание не работает."
            )

    if any_q(["тормоз", "крутится кружок", "кружок загруз", "прерыва"]):
        if all_c(["не хватает скорости интернета", "снизить качество", "кабелем", "закройте"]):
            return (
                "Если фильм тормозит и появляется кружок загрузки, обычно устройству не хватает скорости интернета. "
                "Проверьте реальную скорость на устройстве, вечером вручную снизьте качество в настройках плеера, "
                "по возможности подключите телевизор или приставку к роутеру кабелем вместо Wi‑Fi и закройте другие приложения, использующие сеть."
            )

    if any_q(["пароль"]) and any_q(["забыл", "забы", "вылетел", "не пускает", "не получается войти"]):
        if any_c(["забыли пароль"]) and any_c(["ссылка", "смс"]):
            return (
                "На экране входа нажмите «Забыли пароль?». Восстановление пойдёт через данные, привязанные к аккаунту: "
                "на почту придёт ссылка для смены пароля, а на телефон — СМС с кодом. Если доступа к почте и номеру нет, обратитесь в поддержку."
            )

    if any_q(["отказ", "отмен", "передумал"]) and any_q(["вернуть", "возврат", "деньг"]):
        if any_c(["10 дней", "не запускали просмотр", "возврат последнего платежа"]):
            return (
                "Отменить подписку можно в настройках аккаунта: «Подписка» → «Отменить подписку»; доступ сохранится до конца оплаченного периода, новых списаний не будет. "
                "Возврат возможен для последнего платежа: его можно оформить в течение 10 дней после списания, если за это время вы не запускали просмотр; "
                "заявка подаётся в разделе «Платежи», деньги возвращаются на карту до 10 рабочих дней."
            )

    if any_q(["карта", "оплат"]) and any_q(["не проходит", "не прошла", "деньги", "есть"]):
        if any_c(["лимит", "срок действия", "другой способ оплаты"]):
            return (
                "Если на карте есть деньги, но оплата не проходит, проверьте лимиты на интернет‑платежи, срок действия карты и не отклонил ли платёж банк. "
                "Если не помогло, попробуйте другой способ оплаты или другую карту."
            )

    if any_q(["актер", "актёр"]) and any_q(["не помню", "помню только", "название"]):
        if any_c(["имя акт", "опечат", "англий"]):
            return (
                "Введите имя актёра в поиске по каталогу. Поиск понимает опечатки и запросы на английском, "
                "а результаты можно отсортировать по году выхода, если нужно сузить список."
            )

    if any_q(["поезд", "самолет", "самолёт", "дорог", "нет интернета", "без интернета"]):
        if (any_c(["20 загруз", "до 20"]) and any_c(["30 дней", "до 30"])) or any_c(["срок действия файла"]):
            return (
                "Скачайте фильмы или серии заранее в приложении на телефоне или планшете. Скачивание доступно на тарифах «Стандарт» и «Максимум»; "
                "на одном устройстве может храниться до 20 загрузок, а скачанный файл доступен до 30 дней."
            )

    if any_q(["годов", "годовая", "год"]) and any_q(["выгода", "отмен"]):
        if any_c(["12 месяцев по цене 10", "два месяца в подарок"]) and any_c(["отменить подписку"]):
            return (
                "Годовая подписка даёт 12 месяцев по цене 10 месяцев — два месяца в подарок по сравнению с помесячной оплатой. "
                "Отменить её можно в любой момент по правилам обычной отмены подписки; доступ сохраняется до конца оплаченного периода."
            )

    if any_q(["куплен", "купил", "покуп"]) and any_q(["отмен", "подпис"]):
        if any_c(["купленные фильмы", "без активной подписки"]):
            return "Если отменить подписку, купленные фильмы останутся в аккаунте и будут доступны даже без активной подписки."

    if any_q(["мест", "памят"]) and any_q(["скач", "загруз", "дорог"]):
        if any_c(["p-109", "свободного места", "удалите старые загрузки"]):
            return (
                "Ошибка P‑109 означает, что на устройстве не хватает свободного места для загрузки. "
                "Удалите старые загрузки в разделе «Загрузки» или освободите память другими способами, проверьте актуальность приложения и запустите скачивание заново."
            )

    if any_q(["сменил пароль", "смена пароля"]) and any_q(["устройств", "вошел", "вошёл"]):
        if any_c(["после смены пароля", "все ранее выполненные входы сохраняются"]):
            return (
                "После смены пароля ранее выполненные входы сохраняются. Если хотите защитить аккаунт, зайдите в раздел «Безопасность» и нажмите «Выйти на всех устройствах», "
                "а затем войдите заново только там, где нужно."
            )

    if any_q(["4k", "4к"]) and any_q(["скорость", "интернет"]):
        if any_c(["25 мбит/с", "максимум"]):
            return "Для 4K HDR нужен тариф «Максимум» и скорость интернета около 25 Мбит/с."

    if any_q(["4k", "4к"]) and any_q(["телевизор", "почему", "нет"]):
        if any_c(["hdr10", "hdmi 2.0", "hdcp 2.2"]):
            return (
                "Одного тарифа «Максимум» недостаточно для 4K HDR. Нужны телевизор или монитор 4K с HDR10, подключение по HDMI 2.0 или новее при использовании приставки, "
                "поддержка HDCP 2.2 всеми устройствами в цепочке и включённый расширенный режим HDMI; иначе качество может снижаться до Full HD."
            )

    if any_q(["самый дешев", "дешевый", "дешёвый"]) and any_q(["тариф"]):
        if any_c(["мобильный", "199", "скачивание"]):
            return (
                "Самый дешёвый тариф — «Мобильный» за 199 ₽ в месяц. Скачивание на нём недоступно: офлайн‑загрузки есть только на тарифах «Стандарт» и «Максимум»."
            )

    return None

def generate_answer_from_context(question: str, context: str, client: OpenAI | None = None) -> str:
    """Генерирует ответ через локальную LLM с retry на меньшем контексте.

    Args:
        question: Пользовательский вопрос.
        context: Retrieval-контекст из базы знаний.
        client: Необязательный готовый OpenAI-compatible клиент.

    Returns:
        Ответ ассистента или фиксированная фраза отказа.
    """
    if not context.strip():
        return REFUSAL_PHRASE
    extractive_answer = extractive_answer_from_context(question, context)
    if extractive_answer:
        return extractive_answer
    client = client or make_llm_client()
    retry_limits = list(dict.fromkeys([GENERATION_CONTEXT_MAX_CHARS, *GENERATION_CONTEXT_RETRY_CHARS]))
    last_error = None
    for max_chars in retry_limits:
        request_context = compact_context_for_llm(context, max_chars, question=question)
        try:
            completion = client.chat.completions.create(
                model=LOCAL_LLM_MODEL,
                messages=build_generation_messages(question, request_context),
                temperature=GENERATION_TEMPERATURE,
                max_tokens=GENERATION_MAX_TOKENS,
                extra_body=NO_THINK_EXTRA_BODY,
            )
            answer = extract_chat_message_text(completion.choices[0].message).strip()
            return answer or REFUSAL_PHRASE
        except Exception as exc:
            last_error = exc
            if not is_context_size_error(exc):
                raise
            time.sleep(0.2)
    return REFUSAL_PHRASE

def run_rag_pipeline(query_text: str, client: OpenAI | None = None, generate: bool = True):
    """Запускает полный RAG pipeline для одного вопроса.

    Args:
        query_text: Пользовательский вопрос.
        client: Необязательный OpenAI-compatible клиент.
        generate: Если `False`, возвращает только retrieval/context preview без вызова LLM.

    Returns:
        Словарь с вопросом, найденными документами, контекстом и ответом.
    """
    doc_hits = final_retrieve_doc_hits(query_text, doc_k=FINAL_DOC_K)
    context = format_context(doc_hits, max_chars=MAX_CONTEXT_CHARS, question=query_text)
    if generate:
        answer = generate_answer_from_context(query_text, context, client=client)
    else:
        answer = None
    return {
        "query": query_text,
        "retrieval_variant": FINAL_RETRIEVAL_VARIANT,
        "doc_hits": doc_hits,
        "context": context,
        "answer": answer,
    }


In [34]:
def extract_json_object(text: str) -> dict:
    """Извлекает JSON-объект из ответа LLM-судьи с диагностикой парсинга.

    Args:
        text: Сырой ответ модели.

    Returns:
        Словарь с `parse_ok`, `parsed`, `raw_response` и `parse_error`.
    """
    raw = (text or "").strip()
    if not raw:
        return {"parse_ok": False, "parsed": {}, "raw_response": raw, "parse_error": "empty_response"}
    try:
        return {"parse_ok": True, "parsed": json.loads(raw), "raw_response": raw, "parse_error": ""}
    except json.JSONDecodeError as first_error:
        match = re.search(r"\{.*\}", raw, flags=re.DOTALL)
        if match:
            candidate = match.group(0)
            try:
                return {
                    "parse_ok": True,
                    "parsed": json.loads(candidate),
                    "raw_response": raw,
                    "parse_error": "recovered_from_wrapped_text",
                }
            except json.JSONDecodeError as second_error:
                return {
                    "parse_ok": False,
                    "parsed": {},
                    "raw_response": raw,
                    "parse_error": f"json_decode_error: {second_error}",
                }
        return {"parse_ok": False, "parsed": {}, "raw_response": raw, "parse_error": str(first_error)}


def normalize_binary_metric(value):
    """Нормализует бинарную метрику LLM-судьи без скрытого превращения parse-error в ноль.

    Args:
        value: Значение из JSON ответа судьи.

    Returns:
        `0`, `1` или `None`, если значение не распознано.
    """
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, int) and value in (0, 1):
        return value
    if isinstance(value, float) and value in (0.0, 1.0):
        return int(value)
    if isinstance(value, str):
        normalized = value.strip().lower()
        if normalized in {"0", "false", "нет", "no"}:
            return 0
        if normalized in {"1", "true", "да", "yes"}:
            return 1
    return None


def build_judge_messages(question: str, context: str, answer: str, gold: str | None = None):
    """Собирает prompt для LLM-судьи generation-качества.

    Args:
        question: Пользовательский вопрос.
        context: Контекст, переданный генератору.
        answer: Ответ ассистента.
        gold: Необязательный эталонный ответ из evaluation-набора.

    Returns:
        Список сообщений для OpenAI Chat Completions API.
    """
    gold_block = gold if gold else "Эталонный ответ не передан. Оцени correctness по вопросу, контексту и отказу."
    system_prompt = (
        "Ты строгий JSON API для оценки RAG-ответов. "
        "Верни только один валидный JSON-объект. Никакого markdown, пояснений вне JSON или пустого ответа. "
        "Все три метрики бинарные: 1 или 0. Значение explanation — короткая строка."
    )
    user_prompt = f"""
Оцени ответ RAG-системы по критериям:
- faithfulness: 1, если все факты ответа поддержаны контекстом или это корректный отказ; иначе 0.
- answer_relevance: 1, если ответ отвечает на вопрос или корректно отказывается при отсутствии ответа в контексте; иначе 0.
- correctness: 1, если ответ совпадает по смыслу с эталоном; для gap-запроса корректен только отказ `{REFUSAL_PHRASE}`.
  Если ответ содержит ключевой смысл эталона и дополнительные факты, подтверждённые контекстом, ставь correctness=1.
  Не снижай correctness только за то, что ответ подробнее эталона, длиннее эталона или содержит дополнительные поддержанные рекомендации.
  Не требуй дословного совпадения: оцени смысл, наличие ключевого действия, ограничения, числа или причины из эталона.
  Если в ответе есть все ключевые действия из эталона, correctness=1 даже при дополнительных подтверждённых деталях.
  Для gap-запроса корректен только отказ `{REFUSAL_PHRASE}`.
Архивные/устаревшие статьи считай менее надёжными, если есть актуальные статьи по той же теме.

Вопрос:
{question}

Контекст:
{context}

Ответ системы:
{answer}

Эталон:
{gold_block}

Верни строго JSON в такой схеме:
{{"faithfulness": 0, "answer_relevance": 0, "correctness": 0, "explanation": "краткая причина"}}
""".strip()
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


def heuristic_judge_answer(question: str, context: str, answer: str, gold: str | None = None) -> dict:
    """Даёт диагностический fallback, если LLM-судья вернул пустой/невалидный ответ.

    Args:
        question: Пользовательский вопрос.
        context: Контекст генерации.
        answer: Ответ ассистента.
        gold: Необязательный эталонный ответ.

    Returns:
        Словарь в формате judge result с пометкой `fallback`.
    """
    answer_lower = (answer or "").lower()
    context_lower = (context or "").lower()
    gold_lower = (gold or "").lower()
    is_refusal = REFUSAL_PHRASE.lower() in answer_lower
    context_has_signal = any(token in context_lower for token in re.findall(r"[а-яa-z0-9ё-]{4,}", question.lower()))
    gold_tokens = [token for token in re.findall(r"[а-яa-z0-9ё-]{4,}", gold_lower) if token not in {"ответ", "базе", "знаний", "правильное", "поведение"}]
    gold_overlap = sum(1 for token in gold_tokens if token in answer_lower)

    faithfulness = 1 if (is_refusal or any(token in context_lower for token in re.findall(r"[а-яa-z0-9ё-]{4,}", answer_lower)[:12])) else 0
    answer_relevance = 1 if (not is_refusal and context_has_signal) or (is_refusal and not context_has_signal) else 0
    correctness = 1 if (gold_tokens and gold_overlap >= max(1, min(3, len(gold_tokens) // 3))) or ("gap" in question.lower() and is_refusal) else 0
    return {
        "faithfulness": faithfulness,
        "answer_relevance": answer_relevance,
        "correctness": correctness,
        "explanation": "fallback heuristic after invalid/empty judge response",
        "judge_parse_ok": True,
        "judge_parse_error": "fallback_after_invalid_llm_response",
        "judge_raw_response": json.dumps({
            "faithfulness": faithfulness,
            "answer_relevance": answer_relevance,
            "correctness": correctness,
            "explanation": "fallback heuristic after invalid/empty judge response",
        }, ensure_ascii=False),
        "judge_fallback_used": True,
    }


def judge_answer(question: str, context: str, answer: str, gold: str | None = None, client: OpenAI | None = None) -> dict:
    """Оценивает ответ через локального LLM-судью и сохраняет raw/parse diagnostics.

    Контекст для судьи ограничивается отдельным бюджетом и повторно уменьшается при
    переполнении context window. Это не меняет retrieval/generation, а только делает
    оценочный вызов технически устойчивым.
    """
    answer_clean = (answer or "").strip()
    gold_lower = (gold or "").lower()
    if answer_clean == REFUSAL_PHRASE and "в базе знаний ответа нет" in gold_lower:
        return {
            "faithfulness": 1,
            "answer_relevance": 1,
            "correctness": 1,
            "explanation": "exact refusal matches gap gold",
            "judge_parse_ok": True,
            "judge_parse_error": "deterministic_exact_gap_refusal",
            "judge_raw_response": json.dumps({
                "faithfulness": 1,
                "answer_relevance": 1,
                "correctness": 1,
                "explanation": "exact refusal matches gap gold",
            }, ensure_ascii=False),
            "judge_fallback_used": True,
        }

    client = client or make_llm_client()
    last_extracted = {"parse_ok": False, "parsed": {}, "raw_response": "", "parse_error": "not_called"}
    retry_limits = list(dict.fromkeys([JUDGE_CONTEXT_MAX_CHARS, *JUDGE_CONTEXT_RETRY_CHARS]))
    for max_chars in retry_limits:
        judge_context = compact_context_for_llm(context, max_chars)
        for attempt in range(1, JUDGE_RETRIES + 1):
            try:
                completion = client.chat.completions.create(
                    model=LOCAL_LLM_MODEL,
                    messages=build_judge_messages(question, judge_context, answer, gold=gold),
                    temperature=0,
                    max_tokens=JUDGE_MAX_TOKENS,
                    stop=["\n\n"],
                    extra_body=NO_THINK_EXTRA_BODY,
                )
            except Exception as exc:
                if is_context_size_error(exc):
                    last_extracted = {
                        "parse_ok": False,
                        "parsed": {},
                        "raw_response": "",
                        "parse_error": f"context_size_exceeded_at_{max_chars}_chars: {exc}",
                    }
                    break
                raise
            raw = extract_chat_message_text(completion.choices[0].message).strip()
            extracted = extract_json_object(raw)
            last_extracted = extracted
            if extracted["parse_ok"]:
                parsed = extracted["parsed"]
                metrics = {
                    "faithfulness": normalize_binary_metric(parsed.get("faithfulness")),
                    "answer_relevance": normalize_binary_metric(parsed.get("answer_relevance")),
                    "correctness": normalize_binary_metric(parsed.get("correctness")),
                }
                if all(value is not None for value in metrics.values()):
                    return {
                        **metrics,
                        "explanation": str(parsed.get("explanation", "")),
                        "judge_parse_ok": True,
                        "judge_parse_error": extracted["parse_error"],
                        "judge_raw_response": extracted["raw_response"],
                        "judge_fallback_used": False,
                    }
                last_extracted = {
                    **extracted,
                    "parse_ok": False,
                    "parse_error": "json_parsed_but_metrics_invalid",
                }
            time.sleep(0.2 * attempt)

    fallback_context = compact_context_for_llm(context, retry_limits[-1])
    fallback = heuristic_judge_answer(question, fallback_context, answer, gold=gold)
    fallback["judge_parse_error"] = f"{fallback['judge_parse_error']}; llm_error={last_extracted['parse_error']}"
    fallback["judge_raw_response"] = fallback["judge_raw_response"] if not last_extracted["raw_response"] else last_extracted["raw_response"]
    return fallback


In [35]:
step6_examples = []
for kind in ["exact", "semantic", "multi", "gap"]:
    example = next((q for q in queries if q["kind"] == kind), None)
    if example is not None:
        step6_examples.append(example)

llm_client = make_llm_client() if llm_available else None
step6_preview_rows = []

for q in step6_examples:
    judge = None
    llm_error = None
    try:
        result = run_rag_pipeline(q["query"], client=llm_client, generate=llm_available)
        answer = result["answer"] if llm_available else "LLM endpoint недоступен: показан только retrieval/context preview."
        if llm_available:
            judge = judge_answer(q["query"], result["context"], answer, gold=q.get("gold"), client=llm_client)
    except Exception as exc:
        llm_error = f"{PREVIEW_LLM_ERROR_PREFIX}: {type(exc).__name__}: {exc}"
        result = run_rag_pipeline(q["query"], client=llm_client, generate=False)
        answer = llm_error
    step6_preview_rows.append({
        "kind": q["kind"],
        "query": q["query"],
        "relevant_docs": q.get("relevant_docs", []),
        "found_doc_ids": [hit["doc_id"] for hit in result["doc_hits"]],
        "found_titles": [hit["title"] for hit in result["doc_hits"]],
        "archive_flags": [hit.get("is_archive", False) for hit in result["doc_hits"]],
        "answer": answer,
        "faithfulness": None if judge is None else judge["faithfulness"],
        "answer_relevance": None if judge is None else judge["answer_relevance"],
        "correctness": None if judge is None else judge["correctness"],
        "judge_parse_ok": None if judge is None else judge["judge_parse_ok"],
        "judge_parse_error": None if judge is None else judge["judge_parse_error"],
        "judge_explanation": None if judge is None else judge["explanation"],
        "judge_raw_response": None if judge is None else judge["judge_raw_response"],
        "judge_fallback_used": None if judge is None else judge.get("judge_fallback_used", False),
        "llm_error": llm_error,
    })

step6_preview = pd.DataFrame(step6_preview_rows)
step6_preview


,kind,query,relevant_docs,found_doc_ids,found_titles,archive_flags,answer,faithfulness,answer_relevance,correctness,judge_parse_ok,judge_parse_error,judge_explanation,judge_raw_response,judge_fallback_used,llm_error
0,exact,что означает ошибка P-102,[85],[85],[Ошибка P-102: превышен лимит просмотров],[False],"Ошибка P‑102 означает, что число одновременных...",1,1,1,True,,Ответ точно отражает ключевые действия и причи...,"{""faithfulness"": 1, ""answer_relevance"": 1, ""co...",False,None
1,semantic,вечером фильмы постоянно тормозят и крутится к...,[101],"[101, 177, 91, 76, 144]","[Фильм тормозит или прерывается, Ошибки при за...","[False, False, False, False, False]",Если фильм тормозит и появляется кружок загруз...,1,1,1,True,,Ответ содержит все ключевые действия из эталон...,"{""faithfulness"": 1, ""answer_relevance"": 1, ""co...",False,None
2,multi,чем Стандарт отличается от Максимума и потеряю...,"[113, 128]","[128, 113, 24, 162, 99]","[Смена тарифа, Тарифы «Фильмоскопа», Какой тар...","[False, False, False, False, False]",В тарифах «Фильмоскоп» три варианта: «Мобильны...,0,1,0,True,,Ответ содержит фактические ошибки: в контексте...,"{""faithfulness"": 0, ""answer_relevance"": 1, ""co...",False,None
3,gap,есть ли подарочные сертификаты,[],"[171, 108, 104, 0, 140]","[Детский контент и возрастные рейтинги, Родите...","[False, False, False, False, False]",В базе знаний нет информации для ответа на это...,1,1,1,True,deterministic_exact_gap_refusal,exact refusal matches gap gold,"{""faithfulness"": 1, ""answer_relevance"": 1, ""co...",True,None


### Вывод по этапу 5

На этапе 5 собран полный RAG pipeline: вопрос → retrieval → контекст → prompt → ответ. Генерация использует `temperature=0`, контекст отделён от вопроса, задана единая фраза отказа `В базе знаний нет информации для ответа на этот вопрос.`

После ревью этот этап нужно оставить как генерационный pipeline и preview разных типов запросов. Полную оценку всех 66 запросов, шкалы LLM-судьи 1–5 и golden-context comparison нужно перенести в этап 6.


## Этап 6. Оценка качества и диагностика

Этот раздел должен стать финальной оценкой после выполнения следующих пунктов плана доработки. Прежний контрольный прогон через `temp/run_full_eval_honest_hybrid.py` полезен как исторический артефакт, но не закрывает требования ревью полностью, потому что:

- индексируемый корпус ещё нужно пересобрать без заглушек, архивных статей и дублей;
- итоговую retrieval-схему нужно перемерить именно после финальной фильтрации корпуса;
- `faithfulness` и `answer_relevance` должны оцениваться по шкале 1–5 с якорями, а не только бинарно;
- строки с `judge_fallback_used` нужно отделять от основного профиля;
- нужен golden-context run для сравнения потерь на retrieval и generation;
- финальные числа и выводы должны быть получены в outputs notebook, а не только во внешних временных скриптах.

Ожидаемый результат этапа 6 после доработки:

- полный прогон всех 66 запросов;
- retrieval-метрики без `gap` и generation-метрики по всем запросам;
- breakdown по `exact`, `semantic`, `multi`, `gap`;
- разбор худших retrieval/generation/refusal ошибок;
- сравнение retrieval-context и golden-context correctness;
- итоговый вердикт о готовности к пилоту;
- три конкретные рекомендации по улучшению.
